# Load Libraries

In [ ]:
import sctop as top
import pandas as pd
import numpy as np
import scanpy as sc
import scipy
import gseapy as gs
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.feature_selection import SelectKBest, f_classif
import seaborn as sns
from collections import Counter
import plotly.io as pio
import anndata as ad
import sys
import os
os.chdir('/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/Differentiation/scripts')
sys.path.append('/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/Vilker_Helper_Files/scTOP')
%load_ext autoreload
%autoreload 1
%aimport SimilarityHelper
%aimport TopObject
%aimport CriticalityHelper
%aimport Perturbation
pio.renderers.default = 'notebook'

In [ ]:
# import os
# # os.chdir('../Transdifferentiating-AT2')
# os.getcwd()

In [ ]:
del BasilGenes
counts = sc.read_mtx("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/WIP/Basil/GSM3731532_matrix.mtx")
counts

# Load Bases

In [ ]:
basisCollection = "/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/BasisCollection.csv"
lungMAP = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="LungMAPANOVA",
                                         basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet"]#, "Secretory"]
)
lungMAP2500 = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="LungMAP2500",
                                         basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet", "Secretory"]
)
lungMAP500 = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="LungMAP500",
                                         basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet", "Secretory"]
)
HaberMAP = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="HaberMAP500ANOVA2", #HaberMAP500ANOVA2
                                         basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet", "Secretory", "KRT5-/KRT17+"]
)
HaberMAP500 = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="HaberMAP500",
                                         basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet", "Secretory", "KRT5-/KRT17+"]
)

Adams = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="Adams500", #Adams500ANOVA2
)
Natri = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="Natri2000ANOVA3Alone").drop("Proliferating", axis=1)
# Natri = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="Natri2000ANOVA3")
Natri2000 = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="Natri2000")

In [ ]:
# If getting full LungMAP basis
LungMAP = TopObject.TopObject("LungMAPEpithelial", skipProcess=True, manualInit=False, keep=["Basal", "AT1", "AT2", "Secretory", "Goblet", "Ciliated", "RAS"], maxSamples=5000)
# LungMAP.anndata.var.set_index("_index", inplace=True)
# LungMAP.setup(skipProcess=True, keep=["Basal", "AT1", "AT2", "Secretory", "Goblet", "Ciliated"], maxSamples=3000)
LungMAP.metadata

In [ ]:
LungMAP.annotations.value_counts()

In [ ]:
# LungMAP.testBasis(maxBasisSamples=2000, maxTestSamples=500, includeCriteria=None, trialCount=1, seed=4)
# SimilarityHelper.plotBasisTestConfusionMatrix(LungMAP, title="LungMAP Reclassifications", decimalMode="Clean", #axisFontSize=40
# )
LungMAP.setBasis(maxSamples=2500)
LungMAP.basis.to_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/LungMAPRAS2500.csv")

In [ ]:
LungMAP.annotations.value_counts()

In [ ]:
# habermann = TopObject.TopObject("Habermann", keep=True, skipProcess=True)
# habermann.filter(keep = ["KRT5-/KRT17+"])
# HaberMAP = LungMAP.copy()
# LungMAP.filter(keep=True)
# HaberMAP.mergeWithOther(habermann, includeCriteriaSelf=HaberMAP.annotations.isin(HaberMAP.toKeep), inplace=True)
# LungMAP.mergeWithOther(PPFE, inplace=True)
AdamsMAP = LungMAP.mergeWithOther(kaminski2020, inplace=False, includeCriteriaOther=kaminski2020.annotations=="Aberrant_Basaloid")
AdamsMAP.annotations.value_counts()

In [ ]:
HaberMAP2500.anndata.obs["celltype_level3"].isna().sum()

In [ ]:
# AdamsMAPOG = AdamsMAP.copy()
AdamsMAPOG.filter(maxSamples=2500)
AdamsMAPOG.filterBestGenes(0.2)
AdamsMAPOG.setBasis()

In [ ]:
# LungMAP.anndata.var.set_index("_index", inplace=True)
# Adams500.to_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/Adams500.csv")
natri.basis.to_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/Natri2000ANOVA3Alone.csv")

In [ ]:
genesSelectedFrame, geneProportionFrame = LungMAP.getBestGenes(proportions=[0.1], trialCount=1, seed=1)
geneProportionFrame

In [ ]:
import json
folder = "/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/Vilker_Helper_Files/scTOP/bestGenesResults/"
proportionTestMap = {}
geneProportions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
for proportion in geneProportions:
    proportionTestMap[proportion] = {}

for i in [1, 2, 3, 5, 6]:
    print(i)
    with open(folder + "output" + str(i) + ".json", 'r') as file:
        proportionMapSingle = json.load(file)
        for proportion in geneProportions:
            proportionTestMap[proportion][i] = [proportionMapSingle[str(proportion)][0], proportionMapSingle[str(proportion)][1]]

proportionTestMap[0.3][1][0]

In [ ]:
geneProportions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
trials = [1, 2, 3, 5, 6]
genesSelectedFrame, geneProportionFrame = TopObject.bestGenesAnalysis(proportionTestMap, geneProportions, trials)
geneProportionFrame

In [ ]:
genesSelectedFrame.loc[genesSelectedFrame["Successes"] >= 3]

In [ ]:
HaberMAP.testBasis()

In [ ]:
HaberMAP.getBasisCorrelations(metric="dot")
SimilarityHelper.plotBasisCorrelationMatrix(HaberMAP, figX=8, figY=8, textSize=10,
                                            title="Basis Column Correlations", 
                                            # outFile=None
)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(HaberMAP, figX=8, figY=8, 
                                              # outFile="../../PendingResults/HaberMAP 500 Confusion Matrix.png"
)

In [ ]:
# LungMAP.anndata = LungMAP.anndata[LungMAP.annotations.isin(LungMAP.toKeep)]
# condition = LungMAP.anndata.obs[LungMAP.cellTypeColumn] == "Goblet"
# LungMAP.anndata = ad.AnnData(X=LungMAP.anndata.raw.X, obs=LungMAP.anndata.obs, var=LungMAP.anndata.raw.var)
# LungMAP.anndata.var.set_index("_index", inplace=True)
# LungMAP.anndata.var.index.name = "index"
# LungMAP.anndata.var["gene"] = LungMAP.anndata.var.index
# LungMAP.anndata.obs["lineage_level1"].value_counts()
# LungMAP.setBasis()
# LungMAP.basis.index = LungMAP.anndata.var.index
# LungMAP.basis.index.name = "gene"
LungMAP.basis.to_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/LungMAPANOVA30.csv")

In [ ]:
SimilarityHelper.writeAnnData(LungMAP.anndata, "/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/AnnData/LungMAPEpithelial.h5ad")

# Vannan

In [ ]:
vannan.anndata

In [ ]:
vannan = TopObject.TopObject("Vannan", skipProcess=True)
# vannan.filterAnnObject(keep=True, toKeep=[val for val in vannan.toKeep if val not in ["RASC", "Proliferating Airway"]])
# print(vannan)

In [ ]:
vannan.testBasis()
pd.DataFrame([vannan.testResults[1], vannan.testResults[2]])

In [ ]:
pd.DataFrame([vannan.testResults[1], vannan.testResults[2]]).to_csv("../../PendingResults/Accuracies.csv")

In [ ]:
3-2

In [ ]:
pd.DataFrame([vannan.testResults[1], vannan.testResults[2]]).T

In [ ]:
# vannan.toKeep = names
# vannan.toExclude = ["RASC", "Secretory", "Multiciliated", "Goblet"]
# vannan.filterAnnObject(keep=True, exclude=True)
vannan.metadata

In [ ]:
vannan.metadata['percent_pathology'].value_counts()

In [ ]:
vannan.annObject

In [ ]:
names = []
for i in range(len(valCounts)):
    name = valCounts.index[i]
    count = valCounts.iloc[i]
    if count > 10:
        names.append(name)
names

In [ ]:
vannan.project(habermann.combinedBases["LungMAP"], "HabermannCombined")
# vannan.project(habermann.basis, "Habermann")

In [ ]:
vannan.metadata["sample_affect"].value_counts()

In [ ]:
# Make 2D plot against basis
fig, ax = plt.subplots(1, 1, figsize=(8,8))
axis1 = 'Basal'
axis2 = 'Alveolar type 2'
toExclude = ["AT2", "AT1", "PNEC", "Proliferating Airway"]
# includeCriteria = ~vannan.annotations.isin(toExclude)
includeCriteria = np.logical_and(~vannan.annotations.isin(toExclude), vannan.metadata["sample_affect"] == "More Affected")

SimilarityHelper.plotTwo(vannan.projections["LungMAP"],
         axis1, axis2,
         ax=ax, annotations=vannan.annotations,
         markerSize=40, includeCriteria=includeCriteria
)
plt.legend(bbox_to_anchor=(1,1))
plt.title("Vannan vs LungMAP Basis Similarity Plot")
plt.savefig('../../PendingResults/Vannan vs LungMAP Basal vs AT2 Disease Small.png')
plt.show()

In [ ]:
# Make 2D plot against basis
fig, ax = plt.subplots(1, 1, figsize=(10,10))
axis1 = 'Transitional AT2'
axis2 = 'KRT5-/KRT17+'
# toExclude = ["AT2", "AT1", "PNEC", "Proliferating Airway"]
toInclude = ["Transitional AT2", "KRT5-/KRT17+", "Proliferating AT2", "Basal"]
# includeCriteria = ~vannan.annotations.isin(toExclude)
includeCriteria = vannan.annotations.isin(toInclude)
SimilarityHelper.plotTwo(vannan.projections["HabermannCombined"],
         axis1, axis2,
         ax=ax, annotations=vannan.annotations,
         markerSize=40, includeCriteria=includeCriteria
)
plt.legend(bbox_to_anchor=(1,1))
plt.title("Vannan vs LungMAP + Habermann Basis Similarity Plot")
# plt.tight_layout()
plt.savefig('../../PendingResults/Vannan vs LungMAP + Habermann Transdifferentiating Types Small.png')
plt.show()

In [ ]:
subsetCategory = vannan.metadata['percent_pathology']
subsetNames = sorted([int(val.item()) for val in set(subsetCategory.values)])
subsetNames

In [ ]:
vannan.toKeep

In [ ]:
# So long as the time information was properly uploaded to the TopObject, you're set to run with no changes
toInclude = ["Transitional AT2", "KRT5-/KRT17+", "Proliferating AT2"]
# includeCriteria = vannan.annotations.isin(vannan.toKeep)
includeCriteria = vannan.annotations.isin(toInclude)
fig, axes = SimilarityHelper.plotTwo_multiple(
    vannan, vannan.projections["HabermannCombined"],
    'Transitional AT2', 'KRT5-/KRT17+',
    subsetCategory=subsetCategory, subsetNames=subsetNames,
    includeCriteria=includeCriteria,
    legendFontSize=12,
    similarityBounds=(-0.25, 0.7)
)

fig.suptitle("Vannan vs Habermann + LungMAP Basis Transdifferentiating Types Over Percent Pathology", fontsize=36)
plt.savefig('../../PendingResults/Vannan vs Habermann + LungMAP Basis Transdifferentiating Types Over Percent Pathology.png')
plt.show()

In [ ]:
vannan.project(kathiriya.combinedBases["LungMAP"], "KathiriyaCombined")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
toInclude = ["Transitional AT2", "KRT5-/KRT17+", "Proliferating AT2"]
axis1 = 'ABI1'
axis2 = 'ABI2'
includeCriteria = vannan.annotations.isin(toInclude)
SimilarityHelper.plotTwo(vannan.projections["KathiriyaCombined"],
         axis1, axis2,
         ax=ax, annotations=vannan.annotations,
         includeCriteria=includeCriteria
)
plt.title("Vannan vs LungMAP + Kathiriya Basis Similarity Plot")
plt.savefig('../../PendingResults/ABI1 vs ABI2 Vannan vs LungMAP + Kathiriya Basis')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
toInclude = ["Transitional AT2", "KRT5-/KRT17+", "Proliferating AT2"]
axis1 = 'Transitional AT2'
axis2 = 'KRT5-/KRT17+'
includeCriteria = vannan.annotations.isin(toInclude)
SimilarityHelper.plotTwo(vannan.projections["Habermann"],
         axis1, axis2,
         ax=ax, annotations=vannan.annotations,
         includeCriteria=includeCriteria
)
plt.title("Vannan vs Habermann Basis Similarity Plot")
plt.savefig('../../PendingResults/ABI1 vs ABI2 Vannan vs Habermann Basis')
plt.show()

In [ ]:
vannanDisease = top.process(vannan.df.loc[:, vannan.metadata['sample_affect'] == "More Affected"], average=True)
vannanControl = top.process(vannan.df.loc[:, vannan.metadata['sample_affect'] == "Unaffected"], average=True)

In [ ]:
vannan.setBasis()

In [ ]:
# vannanDisease.rename(columns={0: "More Affected"}, inplace=True)
vannanControl.rename(columns={0: "Unaffected"}, inplace=True)
vannanControl

In [ ]:
# vannanDiseaseDiff = vannanDisease["More Affected"] - vannanControl["Unaffected"]
# vannanDiseaseDiff = pd.DataFrame(vannanDiseaseDiff)
# vannanDiseaseDiff.rename(columns={0: "Disease - Control"}, inplace=True)
# vannanDiseaseDiff.index.name = vannan.basis.index.name
vannanDisease.index.name = vannan.basis.index.name
vannanControl.index.name = vannan.basis.index.name

In [ ]:
diseaseControlVannanBasis = pd.merge(vannan.basis, vannanDisease, on=vannan.basis.index.name, how="inner")
diseaseControlVannanBasis = pd.merge(diseaseControlVannanBasis, vannanControl, on=diseaseControlVannanBasis.index.name, how="inner")
diseaseControlVannanBasis

# Kathiriya

In [ ]:
kathiriya = TopObject.TopObject("Kathiriya", skipProcess=False)
# kathiriya.setBasis()
# kathiriya.combineBases(humanBasis, firstKeep=["ABI1", "ABI2"], name="LungMAP")

In [ ]:
kathiriya.annotations.value_counts()

In [ ]:
# kathiriyaOG = kathiriya.copy()
# # includeCriteria=~kathiriya.annotations.isin(["ABI2", "ABI1", "Ciliated"])
# # includeCriteria = None
# # kathiriya.filter(condition=includeCriteria)
# kathiriya.filterBestGenes(0.3)
# kathiriya.setBasis(includeCriteria=includeCriteria)
kathiriya.basis.to_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/KathiriyaRelabeledANOVA3.csv")

In [ ]:
genesSelectedFrame, geneProportionFrame = kathiriya.getBestGenes(proportions=[0.1, 0.2, 0.3, 0.8, 0.9], trialCount=1) # Made with 3000 no ABI1
geneProportionFrame

In [ ]:
# includeCriteria = None
includeCriteria=~kathiriya.annotations.isin(["Ciliated", "ABI1", "ABI2"])
kathiriya.testBasis(maxBasisSamples=2000, maxTestSamples=500, includeCriteria=includeCriteria, trialCount=5, seed=6)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(kathiriya, title="Kathiriya Basis Test Confusion Matrix", decimalMode="Clean",
                                              outFile="../../Results/BasisTesting/Kathiriya Relabeled Confusion Matrix Downsampled Basis 2000 Test 500 Trials 5.png"
)

In [ ]:
# kathiriya.project(lungMAP, "LungMAP")
# kathiriya.project(lungMAP2500, "LungMAP2500")
# kathiriya.project(HaberMAP, "HaberMAP")
# kathiriya.project(Adams, "Adams")
kathiriya.project(Natri, "Natri")

In [ ]:
# NatriReduced = Natri.drop(['PNEC', 'Proliferating'], axis=1)
# NatriReduced.to_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/Natri2000ANOVA3.csv")

In [ ]:
# kathiriyaSimilarityMap = SimilarityHelper.getMatchingProjections(kathiriya, "Natri")
SimilarityHelper.similarityBoxplot(kathiriyaSimilarityMap,
                                   title="Kathiriya vs Natri", outFile="../../PendingResults/Kathiriya vs Natri Boxplot.png"
)

In [ ]:
# # # newAnnotations = kathiriya.annotations
# newAnnotations = []
# projection = kathiriya.projections["Natri"]
# for cell in kathiriya.annotations.index:
#     scoreAT2 = projection.loc["AT2", cell]
#     scoreBasal = projection.loc["Basal", cell]
#     currentAnno = kathiriya.annotations[cell]
#     if scoreAT2 > 0.5:
#         newAnnotations.append("AEC2s")
#     elif scoreAT2 > 0 and scoreBasal > 0.05 and scoreBasal < 0.3:
#         newAnnotations.append("ABI")
#     elif currentAnno == "ABI2" and scoreBasal > 0.3:
#         newAnnotations.append("Basal")
#     else:
#         newAnnotations.append(currentAnno)
# kathiriya.metadata["newAnnotations"] = newAnnotations
# kathiriya.cellTypeColumn = "newAnnotations"
# kathiriya.metadata = kathiriya.anndata.obs
# kathiriya.annotations = kathiriya.metadata[kathiriya.cellTypeColumn]
kathiriya.annotations.value_counts()

In [ ]:
ax = SimilarityHelper.plotTwo(kathiriya, "Natri", "Basal", "AT2",
                unsupervisedContour=False, DPI=100, maxLabelCount=None, alternateAnnotations=kathiriya.metadata["newAnnotations"],
                # outFile="../../PendingResults/Kathiriya vs LungMAP 2500 Basal vs AT2 Unsupervised 500.png"
)

In [ ]:
ax = SimilarityHelper.plotTwo(kathiriya, "Natri", "Basal", "AT2",
                unsupervisedContour=False, DPI=100, maxLabelCount=None,
                # outFile="../../PendingResults/Kathiriya vs LungMAP 2500 Basal vs AT2 Unsupervised 500.png"
)

In [ ]:
ax = SimilarityHelper.plotTwoMultiple(kathiriya, "Natri", "KRT5-KRT17+", "Basal",
                unsupervisedContour=False, DPI=100, maxLabelCount=500, #gene="FOXJ1",
                # outFile="../../PendingResults/Kathiriya vs Natri SKAR vs Basal.png"
)

In [ ]:
markerGenes = ['SFTPC', 'AGER', 'KRT5', 'KRT17', 'SPRR1A', 'IL32'] # Spread
# markerGenes = ['KRT5', 'KRT17', 'TP63', 'AQP3', 'NGFR', 'DAPL1'] # Basal
# markerGenes = ['CLDN4', 'CDKN2A', 'MMP7', 'KRT17', 'SPRR1A', 'COL1A1', 'PLAUR', 'PLAU'] # SKAR
includeCriteria = None
SimilarityHelper.plotMultipleGenes(kathiriya, "LungMAP", 'Basal', 'AT2', markerGenes, includeCriteria=includeCriteria)

In [ ]:
fig, ax = SimilarityHelper.plot_proportions(newAnnotations, suffixes, suffixSort, rawCounts=False)
plt.title("Kathiriya Source Labels Over Time")
plt.savefig('../Kathiriya/results/Kathiriya Source Labels Time Proportions Plot', bbox_inches='tight')
plt.show()

In [ ]:
# axis1 = "KRT5-KRT17+"
axis1 = "Basal"
axis2 = "AT2"
SimilarityHelper.plotTwoMultiple(kathiriya, "Natri", axis1, axis2,
    plotInRow=False, unsupervisedContour=False, maxLabelCount=500, gene="FOXJ1",
    # outFile="../../PendingResults/Kathiriya vs Natri 2000 ANOVA3 All Days Basal vs AT2.png"
)

In [ ]:
# axis1 = "KRT5-KRT17+"
axis1 = "Basal"
axis2 = "AT2"
# markerGenes = ['CLDN4', 'CDKN2A', 'MMP7', 'IL32', 'SPRR1A', 'COL1A1', 'PLAUR', 'PLAU'] # SKAR
# markerGenes = ['KRT5', 'KRT17', 'KRT8', 'SFTPC', 'ABCA3', 'NAPSA'] # Kathiriya markers, first 2.5 ABI2, second 3.5 ABI1
markerGenes = ['CDHR3', 'FOXJ1', 'DNAH5', 'TUBA'] # Ciliated
SimilarityHelper.plotMultipleGenes(kathiriya, "Natri", axis1, axis2,
    maxLabelCount=None, geneList=markerGenes,
    outFile="../../PendingResults/Kathiriya vs Natri 2000 ANOVA3 All Days Basal vs AT2 Ciliated Markers.png"
)

In [ ]:
axis1 = "Aberrant_Basaloid"
# axis1 = "Basal"
axis2 = "Basal"
SimilarityHelper.plotTwoMultiple(
    kathiriya, "Adams", axis1, axis2, #gene="SPRR1A",
    plotInRow=False, unsupervisedContour=False, DPI=300, maxLabelCount=500,
    # outFile="../../PendingResults/Kathiriya vs Adams 500 SKAR vs Basal.png"
)

In [ ]:
projections_kathiriyaAve = SimilarityHelper.getTimeAveragedProjections(
    completeHumanBasis, kathiriya_df, kathiriya_metadata["celltypes"], 
    suffixes, suffixSort, substituteMap=annotationMap)

In [ ]:
toAssess = "ABI2"
basisKeep = ["Alveolar type 2", "Alveolar type 1", "Basal", "Ciliated", "Goblet", "Secretory"]

fig, ax = plt.subplots(1, 1, figsize = (8, 8))
valueCountsFrame = pd.DataFrame()
for suffix in suffixesSorted:
    valueCountsFrame[suffix] = projections_kathiriyaAve[toAssess + "_" + suffix]
reducedFrame = valueCountsFrame[valueCountsFrame.index.isin(basisKeep)]

ax.stackplot(suffixesSorted, reducedFrame.to_numpy(), labels=reducedFrame.index)
ax.legend(bbox_to_anchor=(1.0, 1.0))
ax.set_xlim(suffixesSorted[0], suffixesSorted[-1])
ax.set_ylim(0, 1)
plt.title("Kathiriya " + toAssess +  " Average Similarity Over Time")
plt.savefig('../Kathiriya/results/Kathiriya ' + toAssess +  ' Average Similarity Over Time Proportions Plot Condensed', bbox_inches='tight')
plt.show()

In [ ]:
basisKeep = ["Alveolar type 2", "Alveolar type 1", "Basal", "Ciliated", "Goblet", "Secretory"]
valueCountsFrame[valueCountsFrame.index.isin(basisKeep)]

In [ ]:
revisedProjections = projections_kathiriya.loc[:, newAnnotations!='Other']
xLabel = 'Respiratory airway secretory'
yLabel = 'Alveolar type 2'
zLabel = 'Basal'
legendTitle = "Kathiriya Annotations"
figureTitle = "Kathiriya vs Human Basis Similarity Plot"
names = newAnnotations[newAnnotations!='Other']
SimilarityHelper.plotThree(revisedProjections, xLabel, yLabel, zLabel, names, figureTitle=figureTitle, legendTitle=legendTitle)

# Kostas

In [ ]:
kostasClusterMap = {}
Kostas5 = sc.read_10x_h5("../KostasHuman/CG5.h5")
Kostas5.var_names_make_unique()
kostasClusterMap[5] = (Kostas5, "Cluster 5")
Kostas6 = sc.read_10x_h5("../KostasHuman/CG6.h5")
Kostas6.var_names_make_unique()
kostasClusterMap[6] = (Kostas6, "Cluster 6")
Kostas7 = sc.read_10x_h5("../KostasHuman/CG7.h5")
Kostas7.var_names_make_unique()
kostasClusterMap[7] = (Kostas7, "Cluster 7")
Kostas8 = sc.read_10x_h5("../KostasHuman/CG8.h5")
Kostas8.var_names_make_unique()
kostasClusterMap[8] = (Kostas8, "Cluster 8")
Kostas12 = sc.read_10x_h5("../KostasHuman/CG12.h5")
Kostas12.var_names_make_unique()
kostasClusterMap[12] = (Kostas12, "Cluster 12")
Kostas13 = sc.read_10x_h5("../KostasHuman/CG13.h5")
Kostas13.var_names_make_unique()
kostasClusterMap[13] = (Kostas13, "Cluster 13")
Kostas14 = sc.read_10x_h5("../KostasHuman/CG14.h5")
Kostas14.var_names_make_unique()
kostasClusterMap[14] = (Kostas14, "Cluster 14 Transitional")

In [ ]:
kostasClusterMapCopy = {}
clusterList = []
clusterNames = []

for clusterNum in kostasClusterMap:
    cluster, celltype = kostasClusterMap[clusterNum]
    cluster.obs["celltype"] = celltype
    kostasClusterMapCopy[clusterNum] = cluster
    clusterList.append(cluster)
    clusterNames.append(celltype)
    
# Kostas14.obs["celltype"] = "Cluster 14 Transitional"
# Kostas7.obs["celltype"] = "Cluster 7 AT2"
# Kostas8.obs["celltype"] = "Cluster 8 AT2"

In [ ]:
Kostas14_df = pd.DataFrame(Kostas14.X.toarray(), index = Kostas14.obs.index , columns = Kostas14.var.index).T

In [ ]:
# Kostas7.obs.index = [f"{name}_2" for name in Kostas7.obs.index]

In [ ]:
combinedKostas = ad.concat(clusterList, join="inner", merge="same")
combinedKostas.obs_names_make_unique()

In [ ]:
Kostas_df = pd.DataFrame(combinedKostas.X.toarray(), index = combinedKostas.obs.index , columns = combinedKostas.var.index).T
kostasData = top.process(Kostas_df)
print("Scored!")


In [ ]:
projections_kostas_no_ras = top.score(noRasBasis, kostasData)
# projections_kostas_kathiriya = top.score(basis, kathiriyaData)
# projections_kostas = top.score(basis, kostasData)

In [ ]:
kostasMetadata = combinedKostas.obs
truePredictedMapKostas = SimilarityHelper.getTopPredictedMap(projections_kostas_no_ras, kostasMetadata, cell_type_column="celltype")
# Counter(truePredictedMapHabermann["KRT5-/KRT17+"])
Counter(truePredictedMapKostas["Cluster 14 Transitional"])


In [ ]:
Counter(truePredictedMapKostas["Cluster 14 Transitional"])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
toKeep = clusterNames
clusters = kostasMetadata["celltype"].values
annotations = SimilarityHelper.setSourceAnnotations(clusters, toKeep, keepAll=True, simplifying=False)
kwargs = SimilarityHelper.setArguments(annotations[annotations!='Other'])
SimilarityHelper.plotTwo(projections_kostas_no_ras.loc[:, annotations!='Other'],
         'Alveolar type 1', 'Alveolar type 2',
         ax=ax, hue=annotations[annotations!='Other'],
         s=40, style=annotations[annotations!='Other'],
         minSimilarity=-0.1, maxSimilarity=0.5
)
plt.legend(bbox_to_anchor=(1,1))
plt.savefig('../../PendingResults/Kostas Human vs LungMAP Basis AT1 vs AT2.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
SimilarityHelper.plotTwo(kathiriya.projections["SimpleLungMAP"], "Alveolar type 1", "Alveolar type 2", ax=ax, annotations=kathiriya.annotations, 
                          unsupervisedContour=True, similarityBounds=(-0.075, 0.45), title="Habermann vs Simplified LungMAP")
SimilarityHelper.compare_populations(ax, kathiriya, kathiriya.projections["SimpleLungMAP"], "Basal", "Alveolar type 2", title="Kathiriya vs LungMAP", similarityBounds=(-0.1, 0.5))
plt.savefig('../../PendingResults/Basal vs AT2 Unsupervised Contour Plot Kathiriya vs Simplified LungMAP.png')
plt.show()

# Habermann

In [ ]:
# habermann = TopObject.TopObject("Habermann", keep=False, skipProcess=False)
# habermann = TopObject.TopObject("Habermann", keep=["AT2", "AT1", "KRT5-/KRT17+", "Basal", "Ciliated", "Transitional AT2", "SCGB3A2+", "SCGB3A2+ SCGB1A1+"], skipProcess=True)
habermann.processed = SimilarityHelper.readProcessed(habermann.df.index, habermann.df.columns, "/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Processed/Habermann.mtx")
habermann.metadata

In [ ]:
SimilarityHelper.writeProcessed(habermann.processed, "/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Processed/Habermann.mtx")

In [ ]:
habermann.annotations.value_counts()

In [ ]:
# habermann.annotations[habermann.metadata["Sample_Name"] == "VUILD48"].value_counts()
# habermann.metadata.loc[habermann.metadata["Diagnosis"] == "Control", :]["Sample_Name"].value_counts()
# haberNames = set(habermann.metadata["Sample_Name"])
# haberOverlap = [name for name in haberNames if name in allNames]
[name for name in haberNames if name not in allNames]
# len(haberOverlap)
# len(haberNames)
# len(set(habermann.metadata["orig.ident"]))
# habermann.metadata[habermann.metadata["Sample_Name"] == "VUHD092"]

In [ ]:
habermann.metadata["Diagnosis"].value_counts()

In [ ]:
habermann.annotations[habermann.metadata["Diagnosis"] == "Control"].value_counts()

In [ ]:
includeCriteria=habermann.annotations.isin(["AT1", "AT2", "Basal", "SKAR", "Differentiating AT2", "Ciliated"])
habermann.testBasis(maxBasisSamples=400, maxTestSamples=500, includeCriteria=includeCriteria, trialCount=5, seed=4)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(habermann, title="Habermann Reclassifications", decimalMode="Clean", axisFontSize=40, figX=14, figY=14,
                                              outFile="../../PendingResults/Habermann Confusion Matrix Downsampled Basis 400 Test 500 Trials 5.png"
)

In [ ]:
# basalMeans = habermann.processed.loc[:, habermann.annotations == "Basal"].mean(axis=1)
# basalMeans = basalMeans.sort_values(ascending=False)

# habermann.setBasis()
# basalDiff = (habermann.basis["Basal"] - habermann.basis["AT2"]).sort_values(ascending=False)
basalCol = lungMAP2500["Basal"]
ciliatedDiff = (basalCol - lungMAP2500["Ciliated"]).sort_values(ascending=False).index
AT1Diff = (basalCol - lungMAP2500["AT1"]).sort_values(ascending=False).index
AT2Diff = (basalCol - lungMAP2500["AT2"]).sort_values(ascending=False).index
# basalDiff = (habermann.basis["Basal"] - habermann.basis["AT2"]).sort_values(ascending=False)

# ranks = {gene: {"Rank": (genes.get_loc(gene) + 30) * (basalDiff.index.get_loc(gene) + 1), "Expression Rank": genes.get_loc(gene), "Diff Rank": basalDiff.index.get_loc(gene)} for gene in basalMeans.index if gene in basalCol.index}
ranks = {gene: {"Expression Rank": basalMeans.index.get_loc(gene), "Ciliated Rank": ciliatedDiff.get_loc(gene), "AT1 Rank": AT1Diff.get_loc(gene), "AT2 Rank": AT2Diff.get_loc(gene)} for gene in basalMeans.index if gene in basalCol.index}
ranks = {gene: {"Rank": ranks[gene]["Ciliated Rank"] + ranks[gene]["AT1 Rank"] + ranks[gene]["AT2 Rank"] + ranks[gene]["Expression Rank"] * 3} | ranks[gene] for gene in ranks.keys()}
rankFrame = pd.DataFrame.from_dict(ranks, orient="index").sort_values("Rank")
rankFrame.iloc[:50]

In [ ]:
# habermann.process()
# habermann.setBasis()
# habermann.anndata.layers["Processed"] = habermann.processed.T
basalDiff = (habermann.basis["Basal"] - habermann.basis["KRT5-/KRT17+"]).sort_values(ascending=True)
# basalDiff = (lungMAP["Basal"] - lungMAP["AT2"]).sort_values(ascending=False)
means = habermann.processed.loc[:, habermann.annotations == "KRT5-/KRT17+"].mean(axis=1)
markerGenes = basalDiff[basalDiff.index.isin(means[means > 0.0025].index)].index[:50]
# markerGenes = basalHits.loc[basalMeans > 0.0025].index[:50]
# markerGenes = rankFrame.index[:50]
# sc.pl.dotplot(habermann.anndata, markerGenes, habermann.cellTypeColumn, layer="Processed", dendrogram=False, return_fig=False)
dp = sc.pl.dotplot(habermann.anndata, markerGenes, habermann.cellTypeColumn, layer="Processed", dendrogram=False, return_fig=True)
dp.savefig("../../PendingResults/Habermann SKAR - Basal Markers (Habermann) Dot Plot.png")

In [ ]:
target = "Basal"
comparator = "Basal"
diseaseColumn = "Diagnosis"
# includeCriteria = habermann.annotations.isin(["AT1", "AT2", "Basal", "SKAR", "Differentiating AT2"])
# includeCriteria = habermann.annotations.isin(["AT1", "AT2", "Basal"])
# includeCriteria = np.logical_or(habermann.annotations == target, np.logical_and(habermann.annotations == comparator, habermann.metadata[diseaseColumn] != "Control"))
# habermann.setBasis(includeCriteria=includeCriteria)
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(habermann.anndata, habermann.cellTypeColumn, target, 
                        individualCompare=True, includeCriteria=None, #basis=habermann.basis
)


habermannTargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, habermann.processed,
                        useBasis=False, includeCriteria=habermann.annotations == target, missesAllowed=5, minimumChange=2, minimumQuantileChange=0.7, maximumPVal=0.1,
                        expressionThreshold=0.0001, diffType="logfoldchanges", checkSurface=False, requireOverexpression=False,
                        # outFile="../../PendingResults/Habermann Differential Cell Surface Genes.csv"
)
habermannTargetDF

In [ ]:
metric = "logfoldchanges"
basalHits = habermannTargetDF.sort_values(metric + " AT1", ascending=False)
# habermannTargetDF["sumScores"] = habermannTargetDF[metric + " AT1"] + habermannTargetDF[metric + " AT2"] + habermannTargetDF[metric + " Ciliated"]
# habermannTargetDF.loc[habermannTargetDF["Successes"] == 2, :].sort_values("sumScores", ascending=False).iloc[:20]
# basalHits = habermannTargetDF.sort_values("sumScores", ascending=False)
# basalHits[habermann.processed.loc[basalHits.index, habermann.annotations == "Basal"].mean(axis=1) > 0.001].sort_values("sumScores", ascending=False)

In [ ]:
# basalHits.loc[basalHits["Average Normalized Expression"] > 0.0001, :]
SimilarityHelper.geneViolinPlot(habermann, "DAPL1")

In [ ]:
newAnnotations = [val if val == "Control" else "Disease" for val in habermann.metadata["Diagnosis"]]
habermann.anndata.obs["diagnosisNew"] = newAnnotations
habermann.setMetadata()

In [ ]:
target = "Disease"
comparator = "Control"
diseaseColumn = "diagnosisNew"
cellState = "Differentiating AT2"
# includeCriteria = habermann.annotations.isin(["AT1", "AT2", "Basal", "SKAR", "Differentiating AT2"])
includeCriteria = np.logical_and(habermann.metadata[diseaseColumn].isin([target, comparator]), habermann.annotations == cellState)
habermann.setBasis(includeCriteria=includeCriteria, annotationColumn=diseaseColumn)
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(habermann.anndata, diseaseColumn, target, 
                        individualCompare=True, includeCriteria=includeCriteria, basis=habermann.basis
)

includeCriteria = np.logical_and(habermann.metadata[diseaseColumn] == target, habermann.annotations == cellState)
habermannTargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, habermann.processed,
                        useBasis=True, includeCriteria=includeCriteria, missesAllowed=3, minimumChange=1, minimumQuantileChange=0.7, maximumPVal=0.1,
                        expressionThreshold=0.0001, diffType="scores", checkSurface=False, requireOverexpression=False,
)
habermannTargetDF

In [ ]:
pre_res = gs.prerank(rnk=habermannTargetDF.loc[habermannTargetDF["Successes"] == 1][comparator],
                     gene_sets='Reactome_Pathways_2024', #KEGG_2026 GO_Biological_Process_2025
                     threads=4,
                     min_size=5,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True, # see what's going on behind the scenes
                    )
# pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/Habermann SKAR vs Disease " + comparator + " Reactome Pathways.csv")
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/Habermann Disease vs Control " + cellState + " Reactome Pathways.csv")
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :]

In [ ]:
# pre_res.res2d.loc[pre_res.res2d["FWER p-val"] < 0.05, :]

In [ ]:
sortedTargetDF = habermannTargetDF.sort_values(by='Successes', ascending=False)
# sortedTargetDF = habermannTargetDF.sort_values(by='scores AT1', ascending=False)
sortedTargetDF.loc[habermannTargetDF["Successes"] > 0]
# sortedTargetDF.loc[sortedTargetDF["scores AT1"] > 2, :]
# diffTableMap['AT1'].loc[diffTableMap['AT1']["pvals_adj"] == 1, :]
# sortedTargetDF.loc[sortedTargetDF["Average Normalized Expression"] > 0.0001, :]

In [ ]:
habermannCopy = habermann.copy()
habermannCopy.filter(conditionList=[habermannCopy.metadata["Diagnosis"].isin(["IPF", "Control"]), habermannCopy.annotations.isin(["AT2", "Transitional AT2", "KRT5-/KRT17+"])])

In [ ]:
target = "IPF"
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(habermannCopy.anndata, "Diagnosis", target, 
                        individualCompare=True, includeCriteria=None
)
# includeCriteria = np.logical_and(includeCriteria, habermannCopy.annotations == target)
# habermann.process()
includeCriteria = habermannCopy.metadata["Diagnosis"] == target
habermannCopyTargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, habermannCopy.processed, 
                        includeCriteria=includeCriteria, missesAllowed=1, minimumChange=1, expressionThreshold=0.00001, requireOverexpression=False, checkSurface=False, diffType="scores",
                        # outFile="../../PendingResults/Habermann Differential Cell Surface Genes.csv"
)
habermannCopyTargetDF

In [ ]:
# habermannCopyTargetDF.sort_values(by='scores Control', ascending=False).iloc[:15]
# habermannCopyTargetDF.loc[habermannCopyTargetDF["Successes"] == 1].sort_values(by='scores Control', ascending=False)
habermannCopyTargetDF.loc["MMP7"]

In [ ]:
# # pre_res = gs.prerank(rnk=basisDiff["diff"],
# pre_res = gs.prerank(rnk=habermannCopyTargetDF.loc[habermannCopyTargetDF["Successes"] == 1]['logfoldchanges Control'],
pre_res = gs.prerank(rnk=habermannCopyTargetDF['logfoldchanges Control'],
                     gene_sets='GO_Biological_Process_2025',
                     # gene_sets='KEGG_2026',
                     threads=4,
                     min_size=5,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True, # see what's going on behind the scenes
                    )
pre_res.res2d.loc[pre_res.res2d["FWER p-val"] < 0.05, :]

In [ ]:
pre_res.res2d.loc[pre_res.res2d["FWER p-val"] < 0.05, :] # Basis diff

In [ ]:
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.05, :] # DEG

In [ ]:
ax = gs.dotplot(pre_res.res2d,
             column="FWER p-val",
             title='IPF vs Control GO Pathways',
             cmap=plt.cm.viridis,
             size=6, # adjust dot size
             figsize=(4,5), cutoff=0.25, show_ring=False)

In [ ]:
enr = gs.enrichr(gene_list=list(habermannCopyTargetDF.sort_values(by='scores Control', ascending=False).index)[:500], gene_sets=["KEGG_2026", "Reactome_2022"], organism='human', outdir=None)
print(enr.results.head(20))

In [ ]:
gene = "ARL4C"
SimilarityHelper.geneViolinPlot(habermann, gene, figX=10, 
                                outFile="../../PendingResults/Habermann " + gene + " Normalized Expression Violin.png"
)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    updatedHabermann, groupby=habermann.cellTypeColumn, standard_scale="var", n_genes=5
)

In [ ]:
# import gseapy as gp
# go_mf = gp.get_library("KEGG_2016", organism="Human")
condition = np.logical_and(genesOfInterest['pvals_adj'] < 0.05, abs(genesOfInterest['logfoldchanges']) > 2)
upCondition = np.logical_and(genesOfInterest['pvals_adj'] < 0.05, genesOfInterest['logfoldchanges'] > 5)
enr_bg = gp.enrichr(gene_list=list(genesOfInterest.loc[upCondition, :]["names"]),
                 gene_sets=['GO_Biological_Process_2025'],
                 # organism='human', # organism argment is ignored because user input a background
                 background=list(genesOfInterest["names"]),
                 outdir=None, # don't write to disk
)
enr_bg.results

In [ ]:
upCondition = np.logical_and(genesOfInterest['pvals_adj'] < 0.05, genesOfInterest['logfoldchanges'] > 5)
genesOfInterest.loc[upCondition, :]

In [ ]:
# pre_res = gp.prerank(genesOfInterest.loc[:, ['names', 'logfoldchanges']], gene_sets='GO_Biological_Process_2025')
pre_res.res2d.head(15)

In [ ]:
names = gp.get_library_name()
names

In [ ]:
habermannGenes = ['Krt17','Prss2','Krt7','Gdf15','Mmp7','Sox4','Tacstd2','Sfn','Ociad2','Mdk','S100a2','Itgb6','Tm4sf1','Krt8','Krt18','Tpm1','Ceacam6','Krt19','Pcsk1n','Ptgs2','Ctse','Hopx','Col1a1','C8orf4','Gprc5a','Pon2','Lamb3','Cldn4','Tram1','Epcam','Fhl2','Itga2','Ccnd2','Tagln','Phlda2','Cst6','Ccnd1','Tnc','Sdc1','Cdkn2a','C19orf33','Lamc2','Pcp4','Lbh','Pmepa1','Zfp36l1','Cdh1','Icam1','Tnfrsf12a','Cd24']
habermann.annotations.value_counts()

In [ ]:
SimilarityHelper.plotPredictivity(habermann, "AT1", testLabel="AT1", 
                                  basis=lungMAP, title="Habermann Predictivity")

In [ ]:
# habermann.combineBases(humanBasis, firstKeep=["Transitional AT2", "KRT5-/KRT17+"], name="LungMAP")
# habermann.combineBases(lungMAP.basis.drop("Secretory", axis=1), firstKeep=["KRT5-/KRT17+"], name="LungMAP")
habermann.combineBases(lungMAP500, firstKeep=["KRT5-/KRT17+"], name="LungMAP500")

## Projections

In [ ]:
# habermann.project(lungMAP, "LungMAP")
# # habermann.project(kaminski2020.basis, "Kaminski")
habermann.project(Natri, "Natri")
# habermann.project(kathiriya.basis, "KathiriyaRelabeled")

In [ ]:
# includeCriteria = habermann.metadata['Diagnosis'] == "IPF"
includeCriteria = None
habermannSimilarityMap = SimilarityHelper.getMatchingProjections(habermann, "LungMAP", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(habermannSimilarityMap, 
        title="Habermann vs LungMAP Similarity Boxplot",
        # outFile="../../PendingResults/Habermann vs LungMAP (Updated, Smallest) Boxplot.png"
)

In [ ]:
# includeCriteria = habermann.metadata['Diagnosis'] == "IPF"
includeCriteria = None
habermannSimilarityMap = SimilarityHelper.getMatchingProjections(habermann, "KathiriyaRelabeled", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(habermannSimilarityMap, 
        title="Habermann vs Kathiriya Similarity Boxplot",
        # outFile="../../PendingResults/Habermann vs KathiriyaRelabeled Boxplot.png"
)

In [ ]:
# toKeep = ["AT1", "AT2", "KRT5-/KRT17+", "Transitional AT2", "SCGB3A2+", "Basal"]
# toKeep = ["AT1", "AT2", "KRT5-/KRT17+", "Proliferating Epithelial Cells", "Transitional AT2", "SCGB3A2+", "SCGB3A2+ SCGB1A1+"]
# includeCriteria = np.logical_and(habermann.annotations != "Basal", habermann.metadata['Diagnosis'] == "IPF")
# includeCriteria = habermann.annotations != "Basal"
includeCriteria = ~habermann.annotations.isin(["MUC5AC+ High", "MUC5B+", "SCGB3A2+", "SCGB3A2+ SCGB1A1+", "Differentiating Ciliated"])
# includeCriteria = np.logical_and(includeCriteria, habermann.metadata['Diagnosis'] == "Control")

ax = SimilarityHelper.plotTwo(habermann, "LungMAP", 'AT1', 'AT2',
                         title="Habermann Projected Onto LungMAP Reference",
                         # outFile="../../PendingResults/Habermann vs LungMAP Basal vs AT2.png",
                         includeCriteria=includeCriteria,
                         # maxLabelCount=500,
                         # unsupervisedContour=True
)

In [ ]:
axis1 = 'Basal'
# axis1 = 'KRT5-KRT17+'
axis2 = 'AT2'
# includeCriteria = habermann.annotations.isin(["AT1", "AT2"])
ax, samples = SimilarityHelper.plotTwo(habermann, "Natri", axis1, axis2,
                         includeCriteria=None, maxLabelCount=500, supervisedContour=False, legendInner=True, getSamples=True,
                         outFile="../../PendingResults/Habermann vs Natri Basal vs AT2 Downsampled 500.png",
)

In [ ]:
# surfaceGeneFile="/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/Transdifferentiation/subcellular_location.tsv"
# cellSurfaceGenes = pd.read_csv(surfaceGeneFile, sep="\t")
candidates = habermann.df.index[habermann.df.index.isin(cellSurfaceGenes.loc[cellSurfaceGenes["Main location"] == "Plasma membrane", :]["Gene name"])]
# means1 = habermann.processed.loc[:, habermann.annotations == "KRT5-/KRT17+"].mean(axis=1)
# means2 = habermann.processed.loc[:, habermann.annotations == "AT2"].mean(axis=1)
# meanDiffs = []
# for i in range(len(means1.index)):
#     meanDiffs.append(abs(means1.iloc[i] - means2.iloc[i]))
# diffsDF = pd.DataFrame({"meanDiffs": meanDiffs}, index=habermann.df.index).sort_values(by="meanDiffs", ascending=False)
# np.quantile(diffsDF.loc[candidates, :]["meanDiffs"], 0.5)
finalCandidates = [val for val in candidates if diffsDF.loc[val, "meanDiffs"] > 0.0008]
winners = []
for gene in finalCandidates:
    if np.quantile(habermann.processed.loc[gene, habermann.annotations == "AT2"], 0.95) < np.quantile(habermann.processed.loc[gene, habermann.annotations == "KRT5-/KRT17+"], 0.5):
        winners.append(gene)
winners

In [ ]:
# markerGenes = ['SFTPC', 'AGER', 'KRT5', 'KRT17', 'SPRR1A', 'IL32'] # Spread
markerGenes = ['KRT5', 'KRT17', 'TP63', 'AQP3', 'NGFR', 'ITGA6'] # Basal
# markerGenes = ['SFTPC', 'SFTPA1', 'SFTPB', 'NKX2-1', 'FABP5'] # AT2
# markerGenes = ['CLDN4', 'CDKN2A', 'MMP7', 'IL32', 'SPRR1A', 'COL1A1', 'PLAUR', 'PLAU'] # SKAR
# markerGenes = ['AQP5', 'TNFRSF9', 'CDH2', 'KISS1R', 'KRT17']
# markerGenes = ['RRAD', 'EFNB1', 'COL17A1', 'DSC2', 'CST6', 'TNFRSF9', 'EDIL3', 'KRT17']

# markerGenes = ['EPHB2', 'KRT17', 'MARCKS', 'CST6', 'CDH2', 'PLPP2']
# keepList = ["iAT1", "iAT2", "ABI1", "ABI2", "iBC 1", "SOX high TGFB-treated"]
# includeCriteria = lauren.annotations.isin(keepList)
SimilarityHelper.plotMultipleGenes(habermann, "Natri", 'KRT5-KRT17+', 'Basal', markerGenes, includeCriteria=habermann.annotations.index.isin(samples), #maxLabelCount=500, 
                                   outFile="../../PendingResults/Habermann vs Natri SKAR vs Basal Basal Markers Downsampled 500.png"
)

In [ ]:
fig, ax = plt.subplots(1, 1)
ax.boxplot([habermann.processed.loc["PLAU", habermann.annotations=="KRT5-/KRT17+"], habermann.processed.loc["PLAU", habermann.annotations!="KRT5-/KRT17+"]])
fig.show()

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(20,20))
axs = ax.flatten()
for i in range(9):
    SimilarityHelper.plotTwo(habermann.projections["SimpleLungMAP"], habermann.annotations,
             'Alveolar type 1', 'Alveolar type 2', ax=axs[i], seed=i, maxLabelCount=500, unsupervisedContour=True, plotMultiple=True)
plt.savefig("../../PendingResults/Habermann vs LungMAP AT1 vs AT2 Downsampled 500.png")
plt.show()

In [ ]:
# clusters = habermann_metadata["celltype"].values
# toKeep = ["AT1", "AT2", "KRT5-/KRT17+", "Transitional AT2", "SCGB3A2+", "Basal"]
# annotations = SimilarityHelper.setSourceAnnotations(clusters, toKeep, keepAll=False)

# revisedProjections = projections_habermann.loc[:, annotations!='Other']
xLabel = 'AT1'
yLabel = 'AT2'
zLabel = 'Secretory'
legendTitle = "Habermann Annotations"
figureTitle = "Habermann 3D Similarity Plot"
names = habermann.annotations
SimilarityHelper.plotThree(habermann.projections["SmallLungMAP"], xLabel, yLabel, zLabel, names, figureTitle=figureTitle, legendTitle=legendTitle)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
# toKeep = ["AT1", "AT2", "KRT5-/KRT17+", "Transitional AT2", "SCGB3A2+", "Basal"]
toKeep = ["KRT5-/KRT17+", "Transitional AT2", "SCGB3A2+", "Basal"]
# toKeep = ["AT1", "AT2", "KRT5-/KRT17+", "Proliferating Epithelial Cells", "Transitional AT2", "SCGB3A2+", "SCGB3A2+ SCGB1A1+"]
axis1 = 'ABI1'
axis2 = 'ABI2'
includeCriteria = habermann.annotations.isin(toKeep)
SimilarityHelper.plotTwo(habermann.projections["KathiriyaCombined"],
         axis1, axis2,
         ax=ax, annotations=habermann.annotations,
         includeCriteria=includeCriteria
)
plt.title("Habermann vs LungMAP + Kathiriya Basis Similarity Plot")
plt.savefig('../../PendingResults/ABI1 vs ABI2 Habermann vs LungMAP + Kathiriya Basis')
plt.show()

In [ ]:
clusters = habermann_metadata["celltype"].values
toKeep = ["AT1", "AT2", "KRT5-/KRT17+", "Proliferating Epithelial Cells", "Transitional AT2", "SCGB3A2+", "Basal"]
annotations = SimilarityHelper.setSourceAnnotations(clusters, toKeep, keepAll=False)

revisedProjections = projections_habermann_kathiriya.loc[:, annotations!='Other']
xLabel = 'Alveolar type 1'
yLabel = 'Alveolar type 2'
zLabel = 'ABI2'
legendTitle = "Habermann Annotations"
figureTitle = "Habermann vs Kathiriya Similarity Plot"
names = annotations[annotations!='Other']
SimilarityHelper.plotThree(revisedProjections, xLabel, yLabel, zLabel, names, figureTitle=figureTitle, legendTitle=legendTitle)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
toKeep = ["KRT5-/KRT17+", "Transitional AT2", "SCGB3A2+", "Basal"]
# toKeep = ["AT1", "AT2", "KRT5-/KRT17+", "Transitional AT2", "SCGB3A2+", "Basal"]
# toKeep = ["AT1", "AT2", "KRT5-/KRT17+", "Proliferating Epithelial Cells", "Transitional AT2", "SCGB3A2+", "SCGB3A2+ SCGB1A1+"]
includeCriteria = habermann.annotations.isin(toKeep)
axis1 = 'Unaffected'
axis2 = 'More Affected'

SimilarityHelper.plotTwo(habermann.projections["DiseaseControlVannanBasis"],
         axis1, axis2,
         ax=ax,
         annotations=habermann.annotations, includeCriteria=includeCriteria
)

plt.title("Habermann vs Vannan + Disease/Control Similarity Plot")
plt.savefig('../../PendingResults/Unaffected vs More Affected Habermann vs Vannan + Disease and Control Small')
plt.show()

In [ ]:
# basisKeep = [col for col in list(tsukui.basis.columns) if col not in ['Alveolar1', 'Alveolar2']]
basisKeep = [col for col in list(diseaseControlVannanBasis.columns) if col in vannan.toKeep or col in ["More Affected", "Unaffected"]]
habermannVannanSimilarityMap = SimilarityHelper.getMatchingProjections(habermann, projections=habermann.projections["DiseaseControlVannanBasis"], sourceKeep=habermann.toKeep, basisKeep=basisKeep)

# Second step: Set basic parameters like plot size and title and generate boxplot
fig, ax = plt.subplots(figsize=(24, 12))
SimilarityHelper.similarityBoxplot(fig, ax, habermannVannanSimilarityMap, title="Habermann vs Vannan Disease + Control Basis Similarity Boxplot")
plt.savefig('../../PendingResults/Habermann vs Vannan Disease + Control Basis Similarity Boxplot.png')
plt.show()

In [ ]:
rng = np.random.default_rng()
aberrant = habermann.metadata.index[habermann.annotations == 'KRT5-/KRT17+']
aberrant = rng.choice(aberrant, size=int(len(aberrant) * 0.5), replace=False)
aberrant

In [ ]:
# notChosenAberrant = habermannCopy.metadata.loc[~habermannCopy.metadata.index.isin(list(aberrant)), :]
# notChosenAberrant
~habermannCopy.metadata.index.isin(aberrant)

In [ ]:
# validSamples = habermann.metadata.index
# validSamples = [sample for sample in habermann.metadata.index if habermann.annotations.loc[sample] != "KRT5-/KRT17+" or sample in aberrant]
len(validSamples)

In [ ]:
habermann.project(habermannCopy.combinedBases["LungMAP"], "TC + LungMAP")

In [ ]:
habermann.project(habermann.combinedBases["LungMAP"], "LungMAP + self")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
# includeCriteria = np.logical_and(habermann.metadata.index.isin(validSamples),
#     # ~habermann.annotations.isin(["KRT5-/KRT17+", "SCGB3A2+", "SCGB3A2+ SCGB1A1+", "Proliferating Epithelial Cells"]))
#     # ~habermann.annotations.isin(["KRT5-/KRT17+", "SCGB3A2+"]))
#     # ~habermann.annotations.isin(["Transitional AT2", "SCGB3A2+"]))
#     ~habermann.annotations.isin(["SCGB3A2+"]))

SimilarityHelper.plotTwo(habermann.projections["LungMAP + self"], habermann.annotations, 
                          "KRT5-/KRT17+", "Alveolar type 2", ax=ax,
                          unsupervisedContour=True, similarityBounds=(-0.2, 0.7),
                          includeCriteria=None, title="Habermann vs Simplified LungMAP with KRT5-/KRT17+ from self")
plt.savefig('../../PendingResults/Habermann Aberrant vs AT2 Unsupervised Contour Plot Habermann vs Simplified LungMAP + Aberrant.png')
plt.show()

In [ ]:
habermann.annotations[habermann.metadata['Diagnosis'] == "IPF"].value_counts()

In [ ]:
# from scipy.spatial.distance import cdist
# includeCriteria = habermann.annotations == "KRT5-/KRT17+"
# proj = habermann.projections["LungMAP"].loc[:, includeCriteria]
# distances = cdist(proj.values.T, proj.values.T, metric="cosine")
np.mean(np.mean(distances, axis=1), axis=0)
# pd.DataFrame(distances, index=proj.columns, columns=proj.columns)
# dist = pd.DataFrame({"dist": distances.mean(axis=1)}, index=proj.columns).sort_values(by="dist", ascending=True)

In [ ]:
# includeCriteria = habermann.metadata['Diagnosis'] == "IPF"
includeCriteria = habermann.annotations.isin(["AT1", "AT2", "Basal", "Ciliated", "Differentiating AT2", "SKAR"])
CriticalityHelper.stateDistancePlot(habermann, "Natri", alternateDF=None, quantile=0.5, method="min", metric="cosine", getNewDistances=False,
                                    includeCriteria=includeCriteria, axisFontSize=36, title="Habermann Distances (scaled by Natri)", 
                                    outFile="../../PendingResults/Habermann vs Natri 2000 ANOVA3 Min Cosine Distance 50% Closest.png"
)

In [ ]:
includeCriteria = habermann.annotations.isin(["AT1", "AT2", "Basal", "Ciliated", "Differentiating AT2", "SKAR"])
# includeCriteria = habermann.metadata['Diagnosis'] == "IPF"

CriticalityHelper.stateDistancePlot(habermann, "Natri", alternateDF=None, quantile=0.5, method="mean", metric="cosine", includeCriteria=includeCriteria, 
                                    title="Habermann vs LungMAP Distances", outFile="../../PendingResults/Habermann vs Natri 2000 ANOVA3 Mean Cosine Distance 50% Closest.png"
)

In [ ]:
CriticalityHelper.selfMeanDistancePlot(stateCloseValuesMap)

In [ ]:
# habermann.setBasis()
res = Perturbation.runProcessedSpaceExperiment(habermann, habermann.basis, "Transitional AT2", "AT1", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
Perturbation.plot_flip_curves(res, "Transitional AT2 → AT1", 
                              # output_file='SCGB3A2_to_AT1.png'
)

In [ ]:
# habermann.setBasis()
res = Perturbation.runProcessedSpaceExperiment(habermann, habermann.basis, "AT1", "AT2", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
Perturbation.plot_flip_curves(res, "AT1 → AT2", 
                              # output_file='SCGB3A2_to_AT1.png'
)

In [ ]:
habermann.setBasis(includeCriteria=habermann.annotations.isin(["AT1", "AT2", "Transitional AT2", "KRT5-/KRT17+", "Basal", "Ciliated"]))
# habermann.setBasis(includeCriteria=habermann.annotations.isin(["AT1", "AT2", "Differentiating AT2", "SKAR", "Basal", "Ciliated"]))
habermannDimensionsMap = Perturbation.getSpaceDimensionsAll(habermann, habermann.basis, maxCells=500, randomState=2, filterToCorrectAtAlpha0=False)
SimilarityHelper.plotSpaceMatrix(habermannDimensionsMap, title="Habermann Perturbations", axisFontSize=34,
    outFile="../../PendingResults/Habermann Perturbation Dimensions Condensed 500 Old Names.png"
)

In [ ]:
habermann.basis

In [ ]:
import math
v = habermann.processed.loc[:, habermann.annotations == "Basal"]
b = habermann.basis["Ciliated"]
thetas = np.arccos(v.T.dot(b))
print("Mean theta: " + str(thetas.mean()))
thetaMean = math.acos(v.mean(axis=1).T.dot(b))
print("Theta mean: " + str(thetaMean))

In [ ]:
import math
v = habermann.processed.loc[:, habermann.annotations == "AT2"]
b = habermann.basis["AT1"]
thetas = np.arccos(v.T.dot(b))
print("Mean theta: " + str(thetas.mean()))
thetaMean = math.acos(v.mean(axis=1).T.dot(b))
print("Theta mean: " + str(thetaMean))

In [ ]:
habermannControl = habermann.copy()
habermannControl.filter(condition=habermann.metadata["Diagnosis"] == "Control")

In [ ]:
# habermannControl.setBasis()
# habermannControl.process()
diff = pd.DataFrame({"Diff": habermannControl.basis["AT2"] - habermannControl.basis["AT1"]})
proj = top.score(diff, habermannControl.processed.loc[:, habermannControl.annotations == "AT1"])
proj.var(axis=1) # AT1 vs AT2 - AT1

In [ ]:
diff = pd.DataFrame({"Diff": habermannControl.basis["AT2"] - habermannControl.basis["AT1"]})
proj = top.score(diff, habermannControl.processed.loc[:, habermannControl.annotations == "AT2"])
proj.var(axis=1) # AT2 vs AT2 - AT1

In [ ]:
diff = pd.DataFrame({"Diff": habermann.basis["AT2"] - habermann.basis["Transitional AT2"]})
proj = top.score(diff, habermann.processed.loc[:, habermann.annotations == "AT2"])
proj.var(axis=1) # AT2 vs AT2 - Transitional AT2

In [ ]:
diff = pd.DataFrame({"Diff": habermann.basis["Transitional AT2"] - habermann.basis["AT2"]})
proj = top.score(diff, habermann.processed.loc[:, habermann.annotations == "Transitional AT2"])
proj.var(axis=1) # Transitional AT2 vs Transitional AT2 - AT2

In [ ]:
diff = pd.DataFrame({"Diff": habermann.basis["Basal"] - habermann.basis["SCGB3A2+"]})
proj = top.score(diff, habermann.processed.loc[:, habermann.annotations == "Basal"])
proj.var(axis=1) # Basal vs Basal - Ciliated

In [ ]:
diff = pd.DataFrame({"Diff": habermann.basis["Basal"] - habermann.basis["SCGB3A2+"]})
proj = top.score(diff, habermann.processed.loc[:, habermann.annotations == "SCGB3A2+"])
proj.var(axis=1) # SCGB3A2+ vs Basal - SCGB3A2+

# Burgess

In [ ]:
burgess = TopObject.TopObject("Burgess", skipProcess=True)
burgess.metadata

In [ ]:
burgess.metadata["orig.ident"]

In [ ]:
burgessDataCopy = burgessData.copy()
burgessDataCopy.index = np.array([gene.upper() for gene in burgessDataCopy.index])
noRasBasisCopy = noRasBasis.copy()
noRasBasisCopy.index = np.array([gene.upper() for gene in noRasBasisCopy.index])
# combinedBasisCopy = combinedBasis.copy()
# combinedBasisCopy.index = np.array([gene.upper() for gene in combinedBasisCopy.index])
projections_burgess = top.score(noRasBasisCopy, burgessDataCopy)
# projections_burgess_combined = top.score(combinedBasisCopy, burgessDataCopy)
projections_burgess

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
# axis1 = 'Transdifferentiating Type II'
# axis2 = 'Cell Cycle Arrest Type II'
axis1 = 'Alveolar type 1'
axis2 = 'Alveolar type 2'
SimilarityHelper.plotTwo(projections_burgess.loc[:, burgessAnnotations!='Other'],
         axis1, axis2,
         ax=ax, hue=burgessAnnotations[burgessAnnotations!='Other'],
         minSimilarity=-0.3, maxSimilarity = 0.3,
         s=40, style=burgessAnnotations[burgessAnnotations!='Other'])
plt.legend(bbox_to_anchor=(1,1))
plt.savefig('../../PendingResults/Burgess vs LungMAP Basis ' + axis1 + ' vs ' + axis2 + '.png')
plt.show()

In [ ]:
projections_burgess_habermann = top.score(habermannBasis, burgessData)
projections_burgess_habermann

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,8))
axis1 = 'Transitional AT2'
axis2 = 'KRT5-/KRT17+'
SimilarityHelper.plotTwo(projections_burgess_habermann.loc[:, burgessAnnotations!='Other'],
         axis1, axis2,
         ax=ax, hue=burgessAnnotations[burgessAnnotations!='Other'],
         minSimilarity=-0.2, maxSimilarity = 0.4,
         s=40, style=burgessAnnotations[burgessAnnotations!='Other'])
plt.legend(bbox_to_anchor=(1,1))
plt.savefig('../../PendingResults/Burgess vs Habermann Basis ' + axis1 + ' vs ' +  'KRT5-KRT17+.png')
plt.show()

In [ ]:
# basisKept = ["AT1", "AT2", "KRT5-/KRT17+", "Proliferating Epithelial Cells", "Transitional AT2", "SCGB3A2+", "SCGB3A2+ SCGB1A1+"]
basisKept = ["AT1", "AT2", "KRT5-/KRT17+", "Transitional AT2", "SCGB3A2+", "SCGB3A2+ SCGB1A1+"]
burgessKept = [celltype for celltype in list(set(burgess_metadata['new_cluster']))]
burgessSimilarityMap = SimilarityHelper.getMatchingProjections(projections_burgess_habermann, burgess_metadata, "new_cluster", basisKept, burgessKept)
fig, ax = plt.subplots(figsize=(20, 8))
plt.title("Burgess vs Habermann Similarity Boxplot")
SimilarityHelper.similarityBoxplot(ax, burgessKept, basisKept, burgessSimilarityMap)
plt.tight_layout()
plt.savefig('../../PendingResults/Burgess vs Habermann Similarity Boxplot.png')
plt.show()

# Kaminski 2020

In [ ]:
kaminski2020 = TopObject.TopObject("Adams2020", keep=True, skipProcess=False)
# kaminski2020 = TopObject.TopObject("Kaminbski2020", keep=True, skipProcess=False, exclude=["Goblet", "Club", "Ciliated"])
kaminski2020.metadata

In [ ]:
# kamFull = sc.read_h5ad("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/AnnData/Kaminski2020Full.h5ad")
kamFull.obs["Manuscript_Identity"].value_counts()

In [ ]:
kaminski2020.metadata["Disease_Identity"].value_counts()

In [ ]:
kaminski2020.annotations[kaminski2020.metadata["Disease_Identity"] == "Control"].value_counts()

In [ ]:
kaminski2020.setBasis()

In [ ]:
# kaminski2020.setAnndata(kaminski2020.anndata[:, kaminski2020.df.index.isin(list(lungMAPSmall.index) + list(habermann.df.index))])
# kaminski2020.anndata
kaminski2020.process()

In [ ]:
target = "SKAR"
comparator = "Basal"
# includeCriteria = habermann.annotations.isin(["ATI", "ATII", "Basal", "SKAR"])
includeCriteria = np.logical_or(kaminski2020.annotations == target, np.logical_and(kaminski2020.annotations == comparator, kaminski2020.metadata["Disease_Identity"] == "Control"))
kaminski2020.setBasis(includeCriteria=includeCriteria)
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(kaminski2020.anndata, kaminski2020.cellTypeColumn, target, 
                        individualCompare=True, includeCriteria=includeCriteria, basis=kaminski2020.basis
)

kaminskiTargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, kaminski2020.processed,
                        useBasis=True, includeCriteria=kaminski2020.annotations == target, missesAllowed=3, minimumChange=1, minimumQuantileChange=0.7, maximumPVal=0.1,
                        expressionThreshold=0.0001, diffType="scores", checkSurface=False, requireOverexpression=False,
)

kaminskiTargetDF

In [ ]:
# # sortedExpression = basalMeans.sort_values(ascending=False)
# # genes = sortedExpression.index
ranks = {gene: {"Rank": (genes.get_loc(gene) + 30) * (basalDiff.index.get_loc(gene) + 1), "Expression Rank": genes.get_loc(gene), "Diff Rank": basalDiff.index.get_loc(gene)} for gene in genes if gene in basalDiff.index}
rankFrame = pd.DataFrame.from_dict(ranks, orient="index").sort_values("Rank")
rankFrame.iloc[:50]

In [ ]:
SimilarityHelper.geneViolinPlot(kaminski2020, "AQP3")

In [ ]:
# kaminski2020.setBasis()
# kaminski2020.anndata.layers["Processed"] = kaminski2020.processed.T
# basalDiff = (kaminski2020.basis["Basal"] - kaminski2020.basis["AT2"]).sort_values(ascending=False)
# basalDiff = (lungMAP2500["Basal"] - lungMAP2500["AT2"]).sort_values(ascending=False)
# basalMeans = kaminski2020.processed.loc[:, kaminski2020.annotations == "Basal"].mean(axis=1)

# markerGenes = basalDiff[basalDiff.index.isin(basalMeans[basalMeans > 0.00325].index)].index[:50]
# markerGenes = basalHits.loc[basalMeans > 0.0025].index[:50]
markerGenes = rankFrame.index[:50]

dp = sc.pl.dotplot(kaminski2020.anndata, markerGenes, kaminski2020.cellTypeColumn, layer="Processed", dendrogram=False, return_fig=True)
dp.savefig("../../PendingResults/Kaminski Basal - AT2 Markers (LungMAP) Dot Plot.png")

In [ ]:
target = "IPF"
comparator = "Control"
diseaseColumn = "Disease_Identity"
cellState = "ATI"
includeCriteria = np.logical_and(kaminski2020.metadata[diseaseColumn].isin([target, comparator]), kaminski2020.annotations == cellState)
kaminski2020.setBasis(includeCriteria=includeCriteria, annotationColumn=diseaseColumn, threshold=150)
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(kaminski2020.anndata, diseaseColumn, target, 
                        individualCompare=True, includeCriteria=includeCriteria, basis=kaminski2020.basis
)

includeCriteria = np.logical_and(kaminski2020.metadata[diseaseColumn] == target, kaminski2020.annotations == cellState)
kaminskiTargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, kaminski2020.processed,
                        useBasis=True, includeCriteria=includeCriteria, missesAllowed=3, minimumChange=1, minimumQuantileChange=0.7, maximumPVal=0.1,
                        expressionThreshold=0.0001, diffType="scores", checkSurface=False, requireOverexpression=False,
)
kaminskiTargetDF

In [ ]:
pre_res = gs.prerank(rnk=kaminskiTargetDF.loc[kaminskiTargetDF["Successes"] == 1][comparator],
                     gene_sets='Reactome_Pathways_2024', #KEGG_2026 GO_Biological_Process_2025
                     threads=4,
                     min_size=5,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True, # see what's going on behind the scenes
                    )
# pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/Adams SKAR vs Control " + comparator + " Reactome Pathways.csv")
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/Adams Disease vs Control " + cellState + " Reactome Pathways.csv")
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :]

In [ ]:
pre_res.res2d.loc[pre_res.res2d["FWER p-val"] < 0.05, :]

In [ ]:
target = "IPF"
# habermann.setBasis()
# diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(kaminski2020Copy.anndata, kaminski2020Copy.cellTypeColumn, target, 
#                         individualCompare=True, includeCriteria=None, #basis=kaminski2020Copy.basis
# )
# diffTableMap
# includeCriteria = np.logical_and(includeCriteria, habermann.annotations == target)
# habermann.process()
includeCriteria = kaminski2020Copy.annotations == target
kaminskiCopyTargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, kaminski2020Copy.processed, 
                        includeCriteria=includeCriteria, missesAllowed=10, minimumChange=0.0001, expressionThreshold=0.00001, checkSurface=False, diffType="logfoldchanges", #useBasis=True,
                        # outFile="../../PendingResults/Habermann Differential Cell Surface Genes.csv"
)
kaminskiCopyTargetDF

In [ ]:
kaminskiCopyTargetDF.loc[kaminskiCopyTargetDF["Successes"] == 1, :]

In [ ]:
ax = gs.dotplot(pre_res.res2d,
             column="FWER p-val",
             title='IPF vs Control GO Pathways',
             cmap=plt.cm.viridis,
             size=4, # adjust dot size
             figsize=(4,5), cutoff=0.05, top_term=10, show_ring=False)

In [ ]:
# genesSelectedFrame, geneProportionFrame = kaminski2020.getBestGenes(proportions=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], trialCount=5)
# geneProportionFrame

In [ ]:
geneProportionFrame

In [ ]:
plt.subplots(1, 1, figsize=(12, 12))
ax = sns.heatmap(geneProportionFrame.astype(float), annot=True, fmt=".2f", cmap='plasma', xticklabels=geneProportionFrame.columns, yticklabels=geneProportionFrame.index,
    annot_kws={"size": 12}, cbar=True)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_xlabel("Trial Number (which train/test split)", fontsize=16)
ax.set_ylabel("Proportion of Genes Used", fontsize=16)
plt.title("Kaminski Basis Trials", fontsize=20)
plt.tight_layout()
plt.savefig("../../PendingResults/Kaminski2020BestGeneTrials.png")

In [ ]:
kaminski2020.testBasis()

In [ ]:
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(kaminski2020.anndata, kaminski2020.cellTypeColumn, "Aberrant_Basaloid", individualCompare=True)
includeCriteria = kaminski2020.annotations == "Aberrant_Basaloid"
targetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, kaminski2020.processed, 
                      includeCriteria=includeCriteria, outFile="Kaminski2020TopSKARSurfaceGenes.csv")
targetDF

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(kaminski2020, title="Kaminski Epithelial Confusion Matrix")

In [ ]:
habermann.PCAs["Kaminski"].feature_names_in_

In [ ]:
kaminski2020.processed.loc[habermann.PCAs["Kaminski"].feature_names_in_, :]

In [ ]:
kaminskiPCA = habermann.PCAs["Kaminski"].transform(kaminski2020.processed.loc[habermann.PCAs["Kaminski"].feature_names_in_, :].T)
kaminskiPCA

In [ ]:
kaminski2020.project(habermann.PCABases["Kaminski"], "HaberPCA", pca=habermann.PCAs["Kaminski"])

## Projections

In [ ]:
# kaminski2020.project(lungMAP, "LungMAP")
kaminski2020.project(Natri, "Natri")
# kaminski2020.project(HaberMAP, "HaberMAP")
# kaminski2020.project(simplifiedHumanBasis, "LungMAP")
# kaminski2020.project(habermann.combinedBases["LungMAP500"], "HaberMAP500")
# kaminskiFull.project(habermann.combinedBases["LungMAP500"], "HaberMAP500")

In [ ]:
kaminski2020.annotations[kaminski2020.metadata['Disease_Identity'] == "IPF"].value_counts()

In [ ]:
# First step: Create map of basis labels to the projections of cells with each source label onto said basis labels
# includeCriteria = kaminskiFull.metadata["Disease_Identity"] != "Control"
includeCriteria = None
kaminskiSimilarityMap = SimilarityHelper.getMatchingProjections(kaminski2020, "HaberMAP", includeCriteria=includeCriteria)

# Second step: Set basic parameters like plot size and title and generate boxplot
SimilarityHelper.similarityBoxplot(kaminskiSimilarityMap, title="Kaminski vs HaberMAP Similarity Boxplot", 
                                   # outFile="../../PendingResults/Kaminski2020 vs HaberMAP 500 ANOVA Boxplot.png"
)

In [ ]:
# includeCriteria = np.logical_and(kaminski2020.annotations != "Ciliated", kaminski2020.metadata['Disease_Identity'] == "Control")
includeCriteria = ~kaminski2020.annotations.isin(["Ciliated", "Goblet", "Club"])
axis1 = 'AT1'
axis2 = 'AT2'

SimilarityHelper.plotTwoObj(kaminski2020, "HaberPCA", axis1, axis2,
                         title="Kaminski vs Habermann Basis AT1 vs AT2 PCA", #outFile="../../PendingResults/Kaminski2020 vs LungMAP AT1 vs AT2 Unsupervised Control.png",
                         includeCriteria=includeCriteria,# maxLabelCount=500,
                         # unsupervisedContour=True
)

In [ ]:
includeCriteria = np.logical_and(kaminski2020.annotations.isin(["ATI", "ATII", "Basal", "SKAR"]), kaminski2020.metadata['Disease_Identity'] != "Control")
axis1 = 'AT2'
axis2 = 'Basal'
# axis2 = 'AT2'

ax = SimilarityHelper.plotTwo(kaminski2020, "LungMAP", axis1, axis2,
                         title="Adams IPF projected on Guo Reference", DPI=300, legendInner=True, axisRenames=("Guo AT2 Cell Score", "Guo Basal Cell Score"),
                         outFile="../../PendingResults/Kaminski2020 IPF vs LungMAP AT2 vs Basal.png",
                         includeCriteria=includeCriteria, maxLabelCount=None,
                         unsupervisedContour=False
)

In [ ]:
includeCriteria = np.logical_and(kaminski2020.annotations.isin(["ATI", "ATII", "Basal", "SKAR"]), kaminski2020.metadata['Disease_Identity'] != "Control")
axis1 = 'KRT5-/KRT17+'
# axis1 = 'Basal'
axis2 = 'Basal'

ax = SimilarityHelper.plotTwo(kaminski2020, "HaberMAP", axis1, axis2,
                         title="Adams IPF projected on HaberMAP", DPI=300, legendInner=True, axisRenames=("Habermann SKAR Cell Score", "Guo Basal Cell Score"),
                         outFile="../../PendingResults/Kaminski2020 IPF vs HaberMAP 500 ANOVA2 SKAR vs Basal.png",
                         includeCriteria=includeCriteria, maxLabelCount=None,
                         unsupervisedContour=False
)

In [ ]:
toKeep = ["Aberrant_Basaloid", "ATI", "ATII", "Basal"]
includeCriteria = kaminski2020.annotations.isin(toKeep)
# axis1 = 'AT1'
axis1 = 'KRT5-/KRT17+'
axis2 = 'AT2'

ax = SimilarityHelper.plotTwo(kaminski2020, "HaberMAP500", axis1, axis2, 
         unsupervisedContour=False, gene="IL32",
         # title="Kaminski 2020 vs LungMAP + Habermann Basal vs AT2",
         # outFile='../../PendingResults/Kaminski 2020 vs LungMAP + Habermann AT1 vs AT2 Unsupervised.png',
         includeCriteria=includeCriteria
)

In [ ]:
axis1 = 'KRT5-/KRT17+'
axis2 = 'Alveolar type 1'
axis3 = 'Alveolar type 2'
legendTitle = "Riemondy Annotations"
figureTitle = "Kaminski2020 vs LungMAP + Habermann 3D Similarity Plot"
SimilarityHelper.plotThree(kaminski2020.projections["LungMAP + Habermann"], axis1, axis2, axis3, kaminski2020.annotations, figureTitle=figureTitle, legendTitle=legendTitle)

In [ ]:
includeCriteria = kaminski2020.metadata['Disease_Identity'] == "Control"
CriticalityHelper.stateDistancePlot(kaminski2020, "LungMAP", quantile=0.5, includeCriteria=includeCriteria, 
                                    title="Kaminski vs LungMAP Distances", outFile="../../PendingResults/Kaminski vs LungMAP Min Euclidean Distance 50% Closest.png")

In [ ]:
# markerGenes = ['SFTPC', 'AGER', 'KRT5', 'KRT17', 'SPRR1A', 'IL32'] # Spread
markerGenes = ['KRT5', 'KRT17', 'TP63', 'AQP3', 'NGFR', 'DAPL1'] # Basal
# markerGenes = ['SFTPC', 'SFTPA1', 'SFTPB', 'NKX2-1', 'FABP5'] # AT2
# markerGenes = ['CLDN4', 'CDKN2A', 'MMP7', 'IL32', 'SPRR1A', 'COL1A1', 'PLAUR', 'PLAU'] # SKAR
# markerGenes = ['DSC2', 'CAVIN1', 'TACSTD2', 'CDH2', 'SVIL']
# markerGenes = ['RRAD', 'EFNB1', 'COL17A1', 'DSC2', 'CST6', 'TNFRSF9', 'EDIL3', 'KRT17']
# markerGenes = ['EFNB1', 'COL17A1', 'PLAUR', 'CST6', 'TNFRSF9']
# markerGenes = ['EPHB2', 'ARL4C', 'MARCKS', 'CST6', 'CDH2', 'PLPP2']
# markerGenes = ['MCAM', 'ICAM1']
# keepList = ["iAT1", "iAT2", "ABI1", "ABI2", "SOX high TGFB-treated"]
# includeCriteria = kaminski2020.annotations.isin(keepList)
SimilarityHelper.plotMultipleGenes(kaminski2020, "Natri", 'KRT5-KRT17+', 'Basal', markerGenes, includeCriteria=None, maxLabelCount=500,
                                   outFile="../../PendingResults/Kaminski vs Natri SKAR vs Basal Basal Markers Downsampled 500.png"
)

In [ ]:
# kaminski2020.setBasis()
res = Perturbation.runProcessedSpaceExperiment(kaminski2020, kaminski2020.basis, "ATII", "ATI", maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
Perturbation.plot_flip_curves(res, "ATII → ATI", 
                              # output_file='SCGB3A2_to_AT1.png'
)

In [ ]:
res = Perturbation.runProcessedSpaceExperiment(kaminski2020, kaminski2020.basis, "ATI", "ATII", maxCells=None, randomState=1, filterToCorrectAtAlpha0=False)
Perturbation.plot_flip_curves(res, "ATI → ATII", 
                              # output_file='SCGB3A2_to_AT1.png'
)

In [ ]:
dimensionsMap = Perturbation.getSpaceDimensionsAll(kaminski2020, kaminski2020.basis, maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
SimilarityHelper.plotSpaceMatrix(dimensionsMap, outFile="../../PendingResults/Kaminski Perturbation Dimensions L.png")

In [ ]:
# kaminski2020.setBasis()
diff = pd.DataFrame({"Diff": kaminski2020.basis["ATII"] - kaminski2020.basis["ATI"]})
proj = top.score(diff, kaminski2020.processed.loc[:, kaminski2020.annotations == "ATI"])
proj.var(axis=1) # AT1 vs AT2 - AT1

In [ ]:
diff = pd.DataFrame({"Diff": kaminski2020.basis["ATII"] - kaminski2020.basis["ATI"]})
proj = top.score(diff, kaminski2020.processed.loc[:, kaminski2020.annotations == "ATII"])
proj.var(axis=1) # AT2 vs AT2 - AT1

# Kaminski 2025

In [ ]:
kaminski = TopObject.TopObject("Kaminski2025AllNucleiInVivoEpithelial", skipProcess=True)

In [ ]:
# kaminski.anndata = sc.read_h5ad(kaminski.filePath)
kaminski.anndata.obs['renamedFinal'].value_counts()

In [ ]:
kaminski.filter(keep=["ATII", "ATI", "Basal", "AberrantBasaloid", "AlvIntermediate"], condition=kaminski.metadata['Disease.Ident'] == "IPF", skipProcess=True)

In [ ]:
rng = np.random.default_rng(seed=1)
randomExclude = kaminski.metadata.index[kaminski.annotations.isin(["ATII", "ATI", "Ciliated"])]
current_IDs = rng.choice(randomExclude, size=int(len(randomExclude) * (0.5)), replace=False)
# cell_IDs = self.metadata[self.annotations == cell_type].index
current_IDs
# vals = [val for val in current_IDs if not val]
# print(len(vals))
# randomExclude =  [sample for sample in kaminski.metadata.index if kaminski.annotations.loc[sample] ]

In [ ]:
kaminski.annotations.value_counts()

In [ ]:
# kaminski.anndata = kaminski.anndata[~kaminski.metadata.index.isin(current_IDs)]
# kaminski.anndata = kaminski.anndata[~kaminski.metadata.index.isin(current_IDs)]
include = kaminski.metadata.index[np.logical_and(~kaminski.annotations.isin(["PNEC", "Secretory", "SecretorySFTPB"]), kaminski.metadata['Disease.Ident'] == "IPF")]
kaminski.anndata = kaminski.anndata[kaminski.metadata.index.isin(include)]
kaminski.setMetadata()
del kaminski.df
kaminski.setDF()
# kaminski.metadata

In [ ]:
kaminski.metadata['Disease.Ident'].value_counts()

In [ ]:
kaminski.setBasis(holdouts=0.9)

In [ ]:
kaminski.testBasis(holdouts=0.2)

In [ ]:
SimilarityHelper.getTestAccuracies(kaminski2025)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(kaminski2025, title="Kaminski Epithelial Confusion Matrix")
# plt.savefig("../Andrea/results/Kaminski Epithelial Confusion Matrix.png")

In [ ]:
# kaminski2025 = TopObject.TopObject("Kaminski2025", skipProcess=True)
kaminski2025.filter(condition=kaminski2025.metadata['Dataset'] == "HabermannEtAl", skipProcess=True)
# kaminski2025.filter(keep=["ATII", "ATI", "Basal", "AberrantBasaloid", "AlvIntermediate"], condition=kaminski2025.metadata['Dataset'] != "Nucleus", skipProcess=True)

In [ ]:
kaminski2025.annotations.value_counts()

In [ ]:
kaminski2025.testBasis(threshold=250)

In [ ]:
kaminski2025.metadata['Dataset']

# Melms

In [ ]:
melms = TopObject.TopObject("Melms", skipProcess=True)
melms.anndata

In [ ]:
melms.metadata.columns

In [ ]:
melms.setAnndata(melms.anndata[:, melms.df.index.isin(lungMAP2500.index)])

In [ ]:
melms.filter(exclude=["Tuft-like", "Cycling epithelial", "ECM-high epithelial"])

In [ ]:
melms.annotations.value_counts()

In [ ]:
melms.newCellTypeCatFromProjection(melms.projections["Habermann LungMAP"], "KRT5-/KRT17+", "AberrantAnnotation")

In [ ]:
melms.testBasis()

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(melmsCopy, figX=10, figY=10)

## Projections

In [ ]:
# melms.project(lungMAP, "LungMAP")
melms.project(LungMAP.basis.drop("Secretory", axis=1), "LungMAPA10")
# melms.project(lungMAP500, "LungMAP500")
# melms.project(lungMAP2500, "LungMAP2500")
# melms.project(habermann.combinedBases["LungMAP500"], "HaberMAP500")

In [ ]:
melms.annotations.value_counts()

In [ ]:
melms.setAnndata(melms.anndata[:, CriticalityHelper.filterDFByGeneVariance(melms.df, proportionKept=0.3).index])

In [ ]:
melmsSimilarityMap = SimilarityHelper.getMatchingProjections(melms, "LungMAP")
SimilarityHelper.similarityBoxplot(melmsSimilarityMap, title="Melms vs LungMAP Similarity Boxplot")

In [ ]:
melmsSimilarityMap = SimilarityHelper.getMatchingProjections(melms, "LungMAPA10")
SimilarityHelper.similarityBoxplot(melmsSimilarityMap, title="Melms vs HaberMAP Similarity Boxplot")

In [ ]:
# axis1 = 'AT1'
axis1 = 'KRT5-/KRT17+'
axis2 = 'AT1'
# includeCriteria = melms.metadata['disease__ontology_label'] != 'normal'
toExclude = ["Airway ciliated", "Airway mucous", "Airway club", "Airway goblet", "Tuft-like"]
includeCriteria = ~melms.annotations.isin(toExclude)
# includeCriteria = melms.annotations.isin(["KRT5-/KRT17+", "AT1", "AT2"])
ax = SimilarityHelper.plotTwo(melms, "HaberMAP500", axis1, axis2,
        includeCriteria=includeCriteria,
        unsupervisedContour=False, maxLabelCount=500,
        # outFile='../../PendingResults/Melms vs LungMAP ' + axis1.replace("/", "") + ' vs ' + axis2 + ' Downsampled 500.png'
)

In [ ]:
axis1 = 'AT1'
axis2 = 'AT2'
# includeCriteria = melms.metadata['disease__ontology_label'] != 'normal'
toExclude = ["Airway ciliated", "Airway mucous", "Airway club", "Airway goblet"]#, "Airway basal"]
includeCriteria = ~melms.annotations.isin(toExclude)
# includeCriteria = melms.annotations.isin(["KRT5-/KRT17+", "AT1", "AT2"])
ax = SimilarityHelper.plotTwo(melms, "LungMAP", axis1, axis2, 
        unsupervisedContour=False, includeCriteria=includeCriteria, gene="CAV1",
        # outFile='../../PendingResults/Melms vs LungMAP ' + axis1.replace("/", "") + ' vs ' + axis2 + '.png'
)

In [ ]:
# markerGenes = ['SFTPC', 'AGER', 'KRT5', 'KRT17', 'SPRR1A', 'IL32'] # Spread
# markerGenes = ['KRT5', 'KRT17', 'TP63', 'AQP3', 'NGFR', 'DAPL1'] # Basal
markerGenes = ['CLDN4', 'CDKN2A', 'MMP7', 'IL32', 'SPRR1A', 'COL1A1', 'PLAUR', 'PLAU'] # SKAR
SimilarityHelper.plotMultipleGenes(melms, "HaberMAP500", 'KRT5-/KRT17+', 'AT1', markerGenes, includeCriteria=includeCriteria)

In [ ]:
melms.scoreContributions["KRT5-/KRT17+"].mean(axis=1).sort_values(ascending=False).head(20).index

In [ ]:
melms.getBasisPredictivity(specificBasis=habermann.combinedBases['Simple LungMAP'])
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
SimilarityHelper.plotPredictivity(ax, melms, "KRT5-/KRT17+", title="Melms Predictivity")
plt.show()

In [ ]:
projection = melms.projections["Habermann LungMAP"]
targetLabel = "KRT5-/KRT17+"
targetCells = []
for sample in melms.df.columns:
    if projection.loc[:, sample].idxmax() == targetLabel and projection.loc[targetLabel, sample] > 0.1:
        targetCells.append(sample)
targetCells

In [ ]:
originalLabels = []
for sample in targetCells:
    originalLabels.append(melms.annotations[sample])
Counter(originalLabels)

In [ ]:
aberrantMetadata = melms.metadata
melms.metadata['AberrantCelltype'] = melms.annotations
melms.metadata['AberrantCelltype'] = melms.metadata['AberrantCelltype'].cat.add_categories(['Aberrant'])

for sample in targetCells:
    melms.metadata.loc[sample, 'AberrantCelltype'] = "Aberrant"
melms.metadata

In [ ]:
# melms.cellTypeColumn = 'AberrantCelltype'
# melms.setup()
melms.processed = melms.df
melms.project(habermann.combinedBases["Simple LungMAP"], "HaberMAP")

In [ ]:
axis1 = 'Alveolar type 1'
axis2 = 'Alveolar type 2'
axis3 = 'KRT5-/KRT17+'

legendTitle = "Melms Annotations"
figureTitle = "Melms vs LungMAP + Habermann 3D Similarity Plot"
SimilarityHelper.plotThree(melms.projections["Habermann LungMAP"], axis1, axis2, axis3, melms.annotations, figureTitle=figureTitle, legendTitle=legendTitle)

# Bharat

In [ ]:
# bharat = sc.read_h5ad("../OutsidePaperObjects/Bharat.h5ad")
# %aimport TopObject
bharat = TopObject.TopObject("Bharat", skipProcess=True)
bharat.anndata

In [ ]:
bharat.annotations.value_counts()

In [ ]:
bharat.annotations[bharat.metadata["Diagnosis"] == "Control (B.)"].value_counts()
# bharat.metadata["Diagnosis"].value_counts()

In [ ]:
bharat.testBasis()

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(bharat, title="Bharat Reclassifications", showPercent=False, cbar=True, square=False, figX=9, figY=9,
                                              outFile="../../Results/BasisTesting/Bharat Basis Confusion Matrix Raw Unnormalized Color CBar.png"
                                              # outFile="../../Results/BasisTesting/Bharat Basis Confusion Matrix Downsampled Basis 1800 Test 10000 Trials 5.png"
)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(bharat, title="Bharat Reclassifications", showPercent=True, square=False, cbar=False, colorNormal=False, #decimalMode="Clean",
                                              # outFile="../../Results/BasisTesting/Bharat Basis Confusion Matrix Clean Decimals.png"
                                              # outFile="../../Results/BasisTesting/Bharat Basis Confusion Matrix Downsampled Basis 1800 Test 10000 Trials 5.png"
)

In [ ]:
# cmNormal.shape
# cm.shape
preprocessing.normalize(cm, axis=1)

In [ ]:
# import math
a = 3
b = 1
c = math.sqrt(a**2 + b**2)
a/c, b/c
# (a/c) / (a/c + b/c)

# from sklearn import preprocessing
# preprocessing.normalize(np.array(((10, 2345, 0, 0, 0, 358, 147, 83, 0), (1, 1, 1, 1, 1, 1, 1, 1, 1))), axis=1)

In [ ]:
# genes90DF = CriticalityHelper.filterDFByGeneVariance(bharatFull.processed, proportionKept=0.1)
# variance = bharatFull.processed.var(axis=1)
lowerQuantile = np.quantile(variance, 0.7)
filteredDF = bharatFull.processed.loc[variance > lowerQuantile, :]
# shared = [gene for gene in filteredDF.index if gene in genesSelectedFrame.index]
bharat.setAnndata(bharat.anndata[:, bharat.df.index.isin(filteredDF.index)])
bharat.anndata

In [ ]:
genesSelectedFrame, geneProportionFrame = bharat.getBestGenes(trialCount=1, proportions=[0.03, 0.06, 0.1, 0.15, 0.2], seed=10)
geneProportionFrame

In [ ]:
genesSelectedFrame

In [ ]:
bharat.metadata["Sample Status"].value_counts()

In [ ]:
# bharat.project(simplifiedHumanBasis, "LungMAP")
# bharat.project(lungMAP, "LungMAP")
# bharat.project(LungMAP.basis.drop("Secretory", axis=1), "LungMAP")
# bharatFull.project(lungMAPSmall, "LungMAP")
# bharat.projections["LungMAP"] = sctopNew.score(lungMAPSmall, bharat.processed)
# bharat.project(HaberMAP, "HaberMAP")
bharat.project(natri.basis, "Natri")

In [ ]:
includeCriteria = bharat.metadata["Sample Status"] == "Control"
bharatSimilarityMap = SimilarityHelper.getMatchingProjections(bharat, "LungMAP", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(bharatSimilarityMap, title="Bharat Control vs LungMAP", 
                                   outFile="../../PendingResults/Bharat Control vs LungMAP Boxplot.png"
)

In [ ]:
# includeCriteria = bharat.metadata["Sample Status"] == "Control"
bharatSimilarityMap = SimilarityHelper.getMatchingProjections(bharat, "Natri", includeCriteria=None)
SimilarityHelper.similarityBoxplot(bharatSimilarityMap, title="Bharat vs Natri", 
                                   # outFile="../../PendingResults/Bharat vs Natri 2000 ANOVA3 Boxplot.png"
)

In [ ]:
condition = np.logical_and(bharat.processed.loc["AGER", :] > 1, bharat.annotations == "KRT17+ KRT5-")
condition = np.logical_and(bharat.processed.loc["SFTPC", :] > 1, condition)
transSamp = bharat.processed.loc[:, condition].columns

In [ ]:
# newCategoryName = "newAnnotations"
# newLabelName = "Differentiating AT2"
# bharat.anndata.obs[newCategoryName] = bharat.anndata.obs[bharat.cellTypeColumn]
# bharat.anndata.obs[newCategoryName] = bharat.anndata.obs[newCategoryName].cat.add_categories([newLabelName])

# for sample in transSamp:
#     bharat.anndata.obs.loc[sample, newCategoryName] = newLabelName

bharat.cellTypeColumn = bharatFull.cellTypeColumn
bharat.annotations = bharat.metadata[bharat.cellTypeColumn]

In [ ]:
includeCriteria = bharat.metadata["Sample Status"] != "Control"
bharat.annotations[includeCriteria].value_counts()

In [ ]:
axis1 = "KRT5-/KRT17+"
# axis1 = "Basal"
axis2 = "AT2"
includeCriteria = bharat.metadata["Sample Status"] != "Controll"
# includeCriteria = bharat.annotations.isin(["KRT17+ KRT5-", "Differentiating AT2"])
includeCriteria = np.logical_and(~bharat.annotations.isin(["Club", "Differentiating ciliated", "Ciliated"]), includeCriteria)
# ax = SimilarityHelper.plotTwo(bharat, "LungMAP", axis1, axis2,
#                             title="Bharat vs LungMAP Basis Similarity Plot", 
#                             outFile="../../PendingResults/Bharat Full vs LungMAP AT1 vs AT2 Unsupervised Downsampled 1000.png",
#                             includeCriteria=includeCriteria, unsupervisedContour=True, 
#                             maxLabelCount=1000
# )
ax = SimilarityHelper.plotTwo(bharat, "HaberMAP500", axis1, axis2,
                            # outFile="../../PendingResults/Bharat vs LungMAP + Habermann Basal vs AT2.png",
                            includeCriteria=includeCriteria,
                            unsupervisedContour=FLSE, 
                            # maxLabelCount=600
)

In [ ]:
axis1 = "KRT5-/KRT17+"
# axis1 = "AT1"
axis2 = "AT2"
# includeCriteria = np.logical_and(bharat.metadata["Sample Status"] != "Control", bharat.annotations == "KRT17+ KRT5-")
ax = SimilarityHelper.plotTwo(bharat, "HaberMAP", axis1, axis2,
                            gene="CDH2", 
                            # title="Bharat vs LungMAP Basis Similarity Plot", 
                            # outFile="../../PendingResults/Bharat vs HaberMAP ARL4C Downsampled 600.png",
                            includeCriteria=includeCriteria, 
                            #maxLabelCount=600, #unsupervisedContour=True,
)

In [ ]:
target = "KRT17+ KRT5-"
includeCriteria = bharat.annotations.isin(["AT1", "AT2", "Basal", "SCGB3A2+", target])
# includeCriteria = None
bharatDiffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(bharat.anndata, bharat.cellTypeColumn, target, 
                        individualCompare=True, includeCriteria=includeCriteria
)
# includeCriteria = bharatFull.annotations.isin(["AT1", "AT2", "Basal", "SCGB3A2+", target])
# diffTableMap2 = CriticalityHelper.getDifferentiallyExpressedGenes(bharatCopy.anndata, bharatCopy.cellTypeColumn, target, 
#                         individualCompare=True, includeCriteria=includeCriteria
# )
includeCriteria = np.logical_and(includeCriteria, bharat.annotations == target)
# includeCriteria = bharat.annotations == target
bharatTargetDF = CriticalityHelper.getCombinedTopGenes(bharatDiffTableMap, bharat.processed, 
                        includeCriteria=includeCriteria, missesAllowed=2, minimumChange=0.5, expressionThreshold=1, checkSurface=False)
bharatTargetDF

In [ ]:
bharatTargetDF.loc[bharatTargetDF["Successes"] == 4].sort_values(by='logfoldchanges AT2', ascending=False)

In [ ]:
# markerGenes = ['SFTPC', 'AGER', 'KRT5', 'KRT17', 'SPRR1A', 'IL32'] # Spread
# markerGenes = ['KRT5', 'KRT17', 'TP63', 'AQP3', 'NGFR', 'DAPL1'] # Basal
# markerGenes = ['SFTPC', 'SFTPA1', 'SFTPB', 'NKX2-1', 'FABP5'] # AT2
# markerGenes = ['CLDN4', 'CDKN2A', 'MMP7', 'IL32', 'SPRR1A', 'COL1A1', 'PLAUR', 'PLAU'] # SKAR
# markerGenes = ['DSC2', 'CAVIN1', 'TACSTD2', 'CDH2', 'SVIL']
markerGenes = ['DSC2', 'MARCKS', 'TACSTD2', 'CDH2', 'CX3CL1', 'KRT17']
includeCriteria = bharat.annotations.isin(["AT1", "AT2", "Basal", "SCGB3A2+", target])
SimilarityHelper.plotMultipleGenes(bharat, "HaberMAP", 'KRT5-/KRT17+', 'AT2', markerGenes, includeCriteria=includeCriteria)

In [ ]:
gene = "ABCC3"
SimilarityHelper.geneViolinPlot(bharat, gene, figX=14, 
                                outFile="../../PendingResults/Bharat " + gene + " Normalized Expression Violin.png"
)

In [ ]:
condition = bharatFull.annotations == target
condition = np.logical_and(condition, bharatFull.processed.loc["SERINC2", :] > 0.1)
# condition = np.logical_and(condition, bharatFull.processed.loc["ARL4C", :] > 0.1)
# condition = np.logical_and(condition, bharatFull.processed.loc["KRT17", :] > 0.1)
# condition = np.logical_and(condition, bharatFull.processed.loc["AGER", :] > 0.1)
bharatFull.processed.loc[:, condition]
# sns.violinplot(bharatFull.processed.loc["SERINC2", condition], inner="quartile")

In [ ]:
bharat.setBasis()
dimensionsMap = Perturbation.getSpaceDimensionsAll(bharat, bharat.basis, maxCells=1000, randomState=1, filterToCorrectAtAlpha0=False)
SimilarityHelper.plotSpaceMatrix(dimensionsMap, outFile="../../PendingResults/Bharat Perturbation Dimensions.png")

In [ ]:
includeCriteria = bharat.annotations.isin(["AT1", "AT2", "Basal", "Ciliated", "KRT17+ KRT5-"])
CriticalityHelper.stateDistancePlot(bharat, "Natri", quantile=0.5, metric="cosine", includeCriteria=includeCriteria, title="Bharat vs Natri Distances", 
                                    outFile="../../PendingResults/Bharat vs Natri Min Cosine Distance 50% Closest.png"
)

In [ ]:
target = "SKAR"
comparator = "Basal"
diseaseColumn = "Diagnosis"
includeCriteria = np.logical_or(bharat.annotations == target, np.logical_and(bharat.annotations == comparator, bharat.metadata[diseaseColumn] != "Control (B.)"))
bharat.setBasis(includeCriteria=includeCriteria)
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(bharat.anndata, bharat.cellTypeColumn, target, 
                        individualCompare=True, includeCriteria=includeCriteria, basis=bharat.basis
)

bharatTargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, bharat.processed,
                        useBasis=True, includeCriteria=bharat.annotations == target, missesAllowed=3, minimumChange=1, minimumQuantileChange=0.7, maximumPVal=0.1,
                        expressionThreshold=0.0001, diffType="scores", checkSurface=False, requireOverexpression=False,
)

bharatTargetDF

In [ ]:
target = "COVID-19"
comparator = "Control (B.)"
diseaseColumn = "Diagnosis"
cellState = "Basal"
includeCriteria = np.logical_and(bharat.metadata[diseaseColumn].isin([target, comparator]), bharat.annotations == cellState)
bharat.setBasis(includeCriteria=includeCriteria, annotationColumn=diseaseColumn, threshold=150)
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(bharat.anndata, diseaseColumn, target, 
                        individualCompare=True, includeCriteria=includeCriteria, basis=bharat.basis
)

includeCriteria = np.logical_and(bharat.metadata[diseaseColumn] == target, bharat.annotations == cellState)
bharatTargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, bharat.processed,
                        useBasis=True, includeCriteria=includeCriteria, missesAllowed=3, minimumChange=1, minimumQuantileChange=0.7, maximumPVal=0.1,
                        expressionThreshold=0.0001, diffType="scores", checkSurface=False, requireOverexpression=False,
)
bharatTargetDF

In [ ]:
pre_res = gs.prerank(rnk=bharatTargetDF.loc[bharatTargetDF["Successes"] == 1][comparator],
                     gene_sets='Reactome_Pathways_2024', #KEGG_2026 GO_Biological_Process_2025
                     threads=4,
                     min_size=5,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True, # see what's going on behind the scenes
                    )
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/Bharat SKAR vs Disease " + comparator + " Reactome Pathways.csv")
# pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/Bharat Disease vs Control " + cellState + " Reactome Pathways.csv")
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :]

# Killian

In [ ]:
killian = TopObject.TopObject("Killian", keep=True, skipProcess=False)
killian.metadata

In [ ]:
killian.annotations.value_counts()

In [ ]:
killian.metadata['diff_day'].value_counts()

In [ ]:
# variances = killian.processed.iloc[:1000, :].var(axis=1)
# np.quantile(variances, 0.4)
# plt.hist(variances)
sns.scatterplot(x=[i * 0.1 for i in range(10)], y=[np.quantile(variances, i * 0.1) for i in range(10)])
# pd.DataFrame(killian.df.var(axis=1), index=killian.df.index)

In [ ]:
killian.df.loc[:, np.logical_and(killian.metadata[killian.timeColumn] == 15, killian.annotations == "Differentiating Alveolar Progenitors")]

In [ ]:
# df = CriticalityHelper.filterDFByGeneVariance(killian.df, threshold=0.1)
varianceMap = {}
# for cluster in killian.sortedCellTypes:
#     print(cluster)
#     varianceMap[cluster] = {}
#     for day in killian.timesSorted:
#         print(day)
#         condition = np.logical_and(killian.metadata[killian.timeColumn] == int(day), killian.annotations == cluster)
#         filt = df.loc[:, condition]
#         if len(filt.columns) == 0:
#             continue
#         varianceMap[cluster][day] = [CriticalityHelper.getClusterCoefficientOfVariation(filt), len(filt.columns)]

for day in killian.timesSorted:
    print(day)
    condition = killian.metadata[killian.timeColumn] == int(day)
    filt = df.loc[:, condition]
    if len(filt.columns) == 0:
        continue
    # varianceMap[day] = [CriticalityHelper.getClusterCoefficientOfVariation(filt), len(filt.columns)]
    # varianceMap[day] = [np.mean(filt.var(axis=1)), len(filt.columns)]
    varianceMap[day] = CriticalityHelper.getGeneEntropy(filt, "ENSG00000168484")


varianceMap

In [ ]:
# [float(varianceMap['Differentiating Alveolar Progenitors'][i][0]) for i in varianceMap['Differentiating Alveolar Progenitors']]
CriticalityHelper.getGeneEntropy(df, "ENSG00000168484")


# PPFE

In [ ]:
PPFE = TopObject.TopObject("PPFE", skipProcess=True)
PPFE.metadata

In [ ]:
PPFE.annotations.value_counts()

In [ ]:
PPFE.filter(keep=["Aberrant_Basaloid"])

In [ ]:
# PPFE.metadata["disease.ident"].value_counts()
PPFE.annotations[PPFE.metadata["disease.ident"] == "CTRL"].value_counts()

In [ ]:
PPFECopy = PPFE.copy()
PPFECopy.cellTypeColumn = "Celltype"
PPFE.annotations = PPFE.metadata[PPFE.cellTypeColumn]
includeCriteria = ~PPFE.annotations.isin(["PNEC", "Ciliated", "AEC_intermediate", "Secretory"])
PPFECopy.filter(condition=includeCriteria)
selector = SelectKBest(score_func=f_classif, k=int(0.1 * len(PPFECopy.df.index)))
trainSelected = selector.fit_transform(PPFECopy.processed.T, PPFECopy.annotations)
del trainSelected
selectedFeatures = PPFECopy.df.index[selector.get_support()]
PPFECopy.setAnndata(PPFECopy.anndata[:, PPFECopy.df.index.isin(list(selectedFeatures))])

In [ ]:
PPFE.testBasis(seed=2, maxBasisSamples=700, maxTestSamples=500, trialCount=5, includeCriteria=PPFE.annotations.isin(["AEC2", "AEC1", "AEC_intermediate", "Aberrant_Basaloid", "Basal", "Secretory", "Ciliated"]))
# PPFE.testBasis(seed=2, maxBasisSamples=1000, maxTestSamples=500, trialCount=5, includeCriteria=PPFE.annotations.isin(["AEC2", "AEC1", "AEC_intermediate", "Basal"]))

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(PPFE, title="PPFE Basis Confusion Matrix", decimalMode="Clean",
                                              # outFile="../../PendingResults/PPFE Confusion Matrix Downsampled Basis 1000 Test 500 Trials 5.png"
)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(PPFE, title="PPFE Basis Confusion Matrix", decimalMode="Clean",
                                              # outFile="../../PendingResults/PPFE Confusion Matrix Downsampled Basis Full 700 Test 500 Trials 5.png"
)

In [ ]:
PPFE.project(lungMAP, "LungMAP")
# PPFE.project(lungMAP2500, "LungMAP2500")
# PPFE.project(lungMAP500, "LungMAP500")
PPFE.project(HaberMAP, "HaberMAP")
PPFE.project(Natri, "Natri")

In [ ]:
includeCriteria = None
PPFESimilarityMap = SimilarityHelper.getMatchingProjections(PPFE, "LungMAP", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(PPFESimilarityMap, title="PPFE vs LungMAP", 
                                   # outFile="../../PendingResults/PPFE vs LungMAP Boxplot.png"
)

In [ ]:
includeCriteria = None
PPFESimilarityMap = SimilarityHelper.getMatchingProjections(PPFE, "LungMAP2500", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(PPFESimilarityMap, title="PPFE vs LungMAP", 
                                   # outFile="../../PendingResults/PPFE vs LungMAP Boxplot.png"
)

In [ ]:
includeCriteria = None
PPFESimilarityMap = SimilarityHelper.getMatchingProjections(PPFE, "HaberMAP500", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(PPFESimilarityMap, title="PPFE vs bharat + LungMAP", 
                                   # outFile="../../PendingResults/PPFE vs LungMAP Boxplot.png"
)

In [ ]:
includeCriteria = ~PPFE.annotations.isin(["PNEC", "Ciliated", "Goblet", "Club", "Secretory"])
# includeCriteria = ~PPFE.annotations.isin(["PNEC", "Ciliated", "Goblet", "AEC2", "AEC_intermediate", "Club", "Basal"])
ax = SimilarityHelper.plotTwo(PPFE, "LungMAP", "AT1", "AT2",
                         maxLabelCount=1500, unsupervisedContour=False, 
                         includeCriteria=includeCriteria, #gene="IL32",
                         title="Ruswich Projected Onto LungMAP Reference", 
                         # outFile="../../PendingResults/PPFE vs LungMAP AT1 vs AT2.png",
)

In [ ]:
includeCriteria = ~PPFE.annotations.isin(["PNEC", "Ciliated", "Goblet", "Club", "Secretory"])
# includeCriteria = ~PPFE.annotations.isin(["PNEC", "Ciliated", "Goblet", "AEC2", "AEC_intermediate", "Club", "Basal"])
ax = SimilarityHelper.plotTwo(PPFE, "HaberMAP", "KRT5-/KRT17+", "AT2",
                         maxLabelCount=500, #unsupervisedContour=True, 
                         includeCriteria=includeCriteria, #gene="IL32",
                         title="Ruswich Projected Onto HaberMAP Reference", 
                         # outFile="../../PendingResults/PPFE vs HaberMAP Basal vs AT2.png",
)

In [ ]:
includeCriteria = ~PPFE.annotations.isin(["PNEC", "Ciliated", "Goblet"])
ax = SimilarityHelper.plotTwo(PPFE, "HaberMAP500", "KRT5-/KRT17+", "AT1",
                         #maxLabelCount=1000, #unsupervisedContour=True,
                         includeCriteria=includeCriteria,
                         title="PPFE Projected Onto Habermann + LungMAP (500 cell/type) Reference", 
                         # outFile="../../PendingResults/PPFE vs HaberMAP500 SKAR vs AT1.png",
)

In [ ]:
includeCriteria = ~PPFE.annotations.isin(["PNEC", "Ciliated", "Goblet"])
# markerGenes = ['SFTPC', 'AGER', 'KRT5', 'KRT17', 'SPRR1A', 'IL32'] # Spread
# markerGenes = ['KRT5', 'KRT17', 'TP63', 'AQP3', 'NGFR', 'DAPL1'] # Basal
markerGenes = ['CLDN4', 'CDKN2A', 'MMP7', 'IL32', 'SPRR1A', 'COL1A1', 'PLAUR', 'PLAU'] # SKAR
SimilarityHelper.plotMultipleGenes(PPFE, "HaberMAP500", 'KRT5-/KRT17+', 'AT2', markerGenes, includeCriteria=includeCriteria)

In [ ]:
samples = PPFE.df.columns[PPFE.annotations == "Aberrant_Basaloid"]
projection = PPFE.projections["HaberMAP500"]
bad2 = [samp for samp in samples if projection.loc["AT1", samp] > projection.loc["KRT5-/KRT17+", samp] + 0.1]
len(bad2)

In [ ]:
newCategoryName = "Celltype_Update1"
PPFE.anndata.obs[newCategoryName] = PPFE.anndata.obs["Celltype"]
# PPFE.anndata.obs[newCategoryName] = PPFE.anndata.obs[newCategoryName].cat.add_categories([newLabelName])

for sample in bad2:
    PPFE.anndata.obs.loc[sample, newCategoryName] = "AEC1"

PPFE.cellTypeColumn = newCategoryName
PPFE.metadata = PPFE.anndata.obs
PPFE.annotations = PPFE.metadata[PPFE.cellTypeColumn]

In [ ]:
target = "SKAR"
comparator = "Basal"
diseaseColumn = "disease.ident"
includeCriteria = np.logical_or(PPFE.annotations == target, np.logical_and(PPFE.annotations == comparator, PPFE.metadata[diseaseColumn] == "CTRL"))
PPFE.setBasis(includeCriteria=includeCriteria)
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(PPFE.anndata, PPFE.cellTypeColumn, target, 
                        individualCompare=True, includeCriteria=includeCriteria, basis=PPFE.basis
)

PPFETargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, PPFE.processed,
                        useBasis=True, includeCriteria=PPFE.annotations == target, missesAllowed=3, minimumChange=1, minimumQuantileChange=0.7, maximumPVal=0.1,
                        expressionThreshold=0.0001, diffType="scores", checkSurface=False, requireOverexpression=False,
)

PPFETargetDF

In [ ]:
target = "PPFE"
comparator = "CTRL"
diseaseColumn = "disease.ident"
cellState = "SKAR"
includeCriteria = np.logical_and(PPFE.metadata[diseaseColumn].isin([target, comparator]), PPFE.annotations == cellState)
PPFE.setBasis(includeCriteria=includeCriteria, annotationColumn=diseaseColumn, threshold=150)
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(PPFE.anndata, diseaseColumn, target, 
                        individualCompare=True, includeCriteria=includeCriteria, basis=PPFE.basis
)

includeCriteria = np.logical_and(PPFE.metadata[diseaseColumn] == target, PPFE.annotations == cellState)
PPFETargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, PPFE.processed,
                        useBasis=True, includeCriteria=includeCriteria, missesAllowed=3, minimumChange=1, minimumQuantileChange=0.7, maximumPVal=0.1,
                        expressionThreshold=0.0001, diffType="scores", checkSurface=False, requireOverexpression=False,
)
PPFETargetDF

In [ ]:
pre_res = gs.prerank(rnk=PPFETargetDF.loc[PPFETargetDF["Successes"] == 1][comparator],
                     gene_sets='Reactome_Pathways_2024', #KEGG_2026 GO_Biological_Process_2025
                     threads=4,
                     min_size=5,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True, # see what's going on behind the scenes
                    )
# pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/PPFE SKAR vs Control " + comparator + " Reactome Pathways.csv")
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/PPFE Disease vs Control " + cellState + " Reactome Pathways.csv")
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :]

# Natri

In [ ]:
# natri = TopObject.TopObject("Natri", skipProcess=True, maxSamples=10000, keepFull=["AT2"])
natri = TopObject.TopObject("Natri", skipProcess=True, maxSamples=10000, keepFull=[])
natri.metadata

In [ ]:
natri.annotations.value_counts()

In [ ]:
natri.metadata["Diagnosis"].value_counts()

In [ ]:
natri.metadata["manual_annotation_1"].value_counts()

In [ ]:
natri.annotations[natri.metadata["Diagnosis"] == "Control"].value_counts()

In [ ]:
natri.filterBestGenes(proportion=0.3)
natri.setBasis(maxSamples=2000)

In [ ]:
# natriMesenchymal = TopObject.TopObject("NatriMesenchymal", skipProcess=True)
includeCriteria=~natriMesenchymal.annotations.isin(["PLIN2+ FB"])
natriMesenchymal.setBasis(annotationColumn="annotation_condensed", includeCriteria=includeCriteria, maxSamples=2000)

In [ ]:
natriMesenchymal.annotations.value_counts()

In [ ]:
newAnno = [val if val != "MyoFB - Activated" else "MyoFB" for val in natriMesenchymal.annotations]
natriMesenchymal.anndata.obs["annotation_condensed"] = newAnno
natriMesenchymal.setMetadata()
natriMesenchymal.metadata["annotation_condensed"].value_counts()

In [ ]:
# includeCriteria=~natriMesenchymal.annotations.isin(["MyoFB", "PLIN2+ FB"])
includeCriteria=~natriMesenchymal.annotations.isin(["PLIN2+ FB"])
natriMesenchymal.testBasis(annotationColumn="annotation_condensed", maxBasisSamples=2000, maxTestSamples=500, includeCriteria=includeCriteria, trialCount=1, seed=41)
SimilarityHelper.plotBasisTestConfusionMatrix(natriMesenchymal, title="Natri Mesenchymal Reclassifications", decimalMode="Clean", #axisFontSize=40
)

In [ ]:
# natriMesenchymal.combineBases(Natri2000, name="NatriEpithelialMesenchymal")
natriMesenchymal.combinedBases["NatriEpithelialMesenchymal"].to_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/NatriEpithelialMesenchymal2000.csv")

In [ ]:
# natri.cellTypeColumn = "manual_annotation_1"
# natri.setMetadata()
includeCriteria = natri.annotations.isin(["AT2", "AT1", "KRT5-/KRT17+", "Basal", "Transitional AT2", "Ciliated"])
# includeCriteria = np.logical_and(includeCriteria, natri.metadata["Diagnosis"] != "Control")
natri.testBasis(maxBasisSamples=1500, maxTestSamples=1000, includeCriteria=includeCriteria, trialCount=5, seed=3)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(natri, title="Natri Reclassifications", decimalMode="Clean", axisFontSize=40,
                                              # outFile="../../Results/BasisTesting/Natri Basis Confusion Matrix Downsampled Basis 1500 Test 1000 Trials 5 Clean.png"
)

In [ ]:
# natri.project(lungMAP, "LungMAP")
natri.project(HaberMAP, "HaberMAP")
# natri.project(Adams, "Adams")

In [ ]:
includeCriteria = None
NatriSimilarityMapLungMAP = SimilarityHelper.getMatchingProjections(natri, "LungMAP", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(NatriSimilarityMapLungMAP, title="Natri vs LungMAP", 
                                   outFile="../../PendingResults/Natri vs LungMAP Boxplot.png"
)

In [ ]:
natri.projections["HaberMAP"] = natri.projections["HaberMAP"].rename(index={'KRT5-/KRT17+': 'SKAR'})
natri.projections["HaberMAP"]

In [ ]:
includeCriteria = ~natri.annotations.isin(["PNEC", "Proliferating", "Goblet"])
# NatriSimilarityMapHaberMAP = SimilarityHelper.getMatchingProjections(natri, "HaberMAP", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(NatriSimilarityMapHaberMAP, title="Natri projections onto Habermann and Guo Reference", basisKeep=["AT1", "AT2", "Basal", "Ciliated", "SKAR", "Secretory"], testKeep=["Basal", "Ciliated", "Secretory", "AT1", "AT2", "SKAR"],
                                   outFile="../../PendingResults/Natri vs HaberMAP Boxplot.png"
)

In [ ]:
includeCriteria = ~natri.annotations.isin(["PNEC", "Proliferating", "Goblet"])
NatriSimilarityMapHaberMAP = SimilarityHelper.getMatchingProjections(natri, "HaberMAP", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(NatriSimilarityMapHaberMAP, title="Natri projections onto Habermann and Guo Reference", basisKeep=["AT1", "AT2", "Basal", "Ciliated", "SKAR", "Secretory"], testKeep=["Basal", "Ciliated", "Secretory", "AT1", "AT2", "SKAR"],
                                   # outFile="../../PendingResults/Natri vs HaberMAP Boxplot.png"
)

In [ ]:
includeCriteria = ~natri.annotations.isin(["PNEC", "Secretory", "Proliferating"])
includeCriteria = np.logical_and(includeCriteria, natri.metadata["Diagnosis"] != "Control")
ax = SimilarityHelper.plotTwo(natri, "LungMAP", "AT1", "AT2",
                         maxLabelCount=9500, unsupervisedContour=False, alpha=1, DPI=300, legendInner=True,
                         includeCriteria=includeCriteria, axisRenames=("Guo AT1 Cell Score", "Guo AT2 Cell Score"),
                         title="Natri Projected Onto Guo Reference", 
                         outFile="../../PendingResults/Natri vs LungMAP AT1 vs AT2 Disease Downsampled 9500.png",
)

In [ ]:
includeCriteria = ~natri.annotations.isin(["PNEC", "Secretory"])
# includeCriteria = natri.annotations.isin(["AT2", "Proliferating"])
markers = ["CLDN4", "IL32", "MMP7", "KRT17", "KRT8", "SCGB3A2", "CEACAM6"]
ax = SimilarityHelper.plotMultipleGenes(natri, "LungMAP", "AT1", "AT2", markers,
                         maxLabelCount=2000, #unsupervisedContour=True, 
                         includeCriteria=includeCriteria,
                         title="Natri Projected Onto LungMAP Reference", 
                         # outFile="../../PendingResults/Natri vs LungMAP AT1 vs AT2 Downsampled 2000.png",
)

In [ ]:
includeCriteria = ~natri.annotations.isin(["PNEC", "Secretory"])
# includeCriteria = natri.annotations.isin(["AT2", "Proliferating"])
# markers = ["CLDN4", "IL32", "MMP7", "KRT17", "KRT8", "SCGB3A2", "CEACAM6"]
markers = ["COL17A1", "COL1A1", "AGER", "SFTPC"]
ax = SimilarityHelper.plotMultipleGenes(natri, "HaberMAP", "KRT5-/KRT17+", "AT2", markers,
                         maxLabelCount=2000, #unsupervisedContour=True, 
                         includeCriteria=includeCriteria,
                         title="Natri Projected Onto LungMAP Reference", 
                         # outFile="../../PendingResults/Natri vs LungMAP AT1 vs AT2 Downsampled 2000.png",
)

In [ ]:
# includeCriteria = ~natri.annotations.isin(["PNEC", "Secretory"])
# includeCriteria = ~natri.annotations.isin(["PNEC", "Secretory", "Proliferating", "Ciliated"])
# includeCriteria = np.logical_and(includeCriteria, natri.metadata["Diagnosis"] != "Control")
axis1 = "KRT5-/KRT17+"
# axis1 = "AT1"
axis2 = "AT2"
ax = SimilarityHelper.plotTwo(natri, "HaberMAP", axis1, axis2,
                         maxLabelCount=1000, unsupervisedContour=True, axisRenames=("Habermann SKAR Cell Score", "Guo AT2 Cell Score"),
                         includeCriteria=None, DPI=100, legendInner=True, gene="RTEL1"
                         title="Natri projected on HaberMAP Reference", 
                         # outFile="../../PendingResults/Natri vs HaberMAP SKAR vs AT2 Disease Downsampled 1000 Unsupervised.png",
)

In [ ]:
includeCriteria = ~natri.annotations.isin(["PNEC", "Secretory"])
includeCriteria = np.logical_and(includeCriteria, natri.metadata["Diagnosis"] != "Control")
# axis1 = "KRT5-/KRT17+"
ax = SimilarityHelper.plotTwo(natri, "Adams", "Aberrant_Basaloid", "ATII",
                         maxLabelCount=2000, #unsupervisedContour=True, 
                         includeCriteria=includeCriteria,
                         title="Natri Projected Onto Adams Reference", 
                         outFile="../../PendingResults/Natri vs Adams 500 ANOVA2 SKAR vs ATII Disease Downsampled 2000.png",
)

In [ ]:
includeCriteria = natri.annotations.isin(["AT1", "AT2", "Basal", "Ciliated", "KRT5-KRT17+"])
CriticalityHelper.stateDistancePlot(natri, "HaberMAP", quantile=0.5, metric="cosine", includeCriteria=includeCriteria, title="Natri vs HaberMAP Distances", 
                                    outFile="../../PendingResults/Natri vs HaberMAP Min Cosine Distance 50% Closest.png"
)

In [ ]:
target = "SKAR"
comparator = "AT2"
diseaseColumn = "Diagnosis"
includeCriteria = np.logical_or(natri.annotations == target, np.logical_and(natri.annotations == comparator, natri.metadata[diseaseColumn] == "Control"))
natri.setBasis(includeCriteria=includeCriteria)
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(natri.anndata, natri.cellTypeColumn, target, 
                        individualCompare=True, includeCriteria=includeCriteria, basis=natri.basis
)

natriTargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, natri.processed,
                        useBasis=True, includeCriteria=natri.annotations == target, missesAllowed=3, minimumChange=1, minimumQuantileChange=0.7, maximumPVal=0.1,
                        expressionThreshold=0.0001, diffType="scores", checkSurface=False, requireOverexpression=False,
)

natriTargetDF

In [ ]:
target = "IPF"
comparator = "Control"
diseaseColumn = "Diagnosis"
cellState = "SKAR"
includeCriteria = np.logical_and(natri.metadata[diseaseColumn].isin([target, comparator]), natri.annotations == cellState)
natri.setBasis(includeCriteria=includeCriteria, annotationColumn=diseaseColumn, threshold=150)
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(natri.anndata, diseaseColumn, target, 
                        individualCompare=True, includeCriteria=includeCriteria, basis=natri.basis
)

includeCriteria = np.logical_and(natri.metadata[diseaseColumn] == target, natri.annotations == cellState)
natriTargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, natri.processed,
                        useBasis=True, includeCriteria=includeCriteria, missesAllowed=3, minimumChange=1, minimumQuantileChange=0.7, maximumPVal=0.1,
                        expressionThreshold=0.0001, diffType="scores", checkSurface=False, requireOverexpression=False,
)
natriTargetDF

In [ ]:
pre_res = gs.prerank(rnk=natriTargetDF.loc[natriTargetDF["Successes"] == 1][comparator],
                     gene_sets='Reactome_Pathways_2024', #KEGG_2026 GO_Biological_Process_2025
                     threads=4,
                     min_size=5,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True, # see what's going on behind the scenes
                    )
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/Natri SKAR vs Control " + comparator + " Reactome Pathways.csv")
# pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/Natri Disease vs Control " + cellState + " Reactome Pathways.csv")
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :]

# Dietrich

In [ ]:
# dietrich = TopObject.TopObject("Dietrich", skipProcess=True, maxSamples=10000)
dietrich = TopObject.TopObject("Dietrich", skipProcess=True, maxSamples=20000, keep=["AT1", "AT2", "Basal", "Aberrant basaloid", "Goblet", "RASC", "Club", "MCC"])
dietrich.metadata

In [ ]:
dietrich.annotations.value_counts()

In [ ]:
dietrich.metadata["diagnosis"].value_counts()

In [ ]:
dietrich.annotations[dietrich.metadata["diagnosis"] == "IPF"].value_counts()

In [ ]:
includeCriteria = dietrich.annotations.isin(["AT2", "AT1", "Aberrant basaloid", "Basal", "MCC", "Goblet", "RASC"])
dietrichProc, dietrichGenes = dietrich.filterBestGenes(0.2, startFromDF=True, inplace=False, maxSamples=5000, includeCriteria=includeCriteria)
dietrichProc

In [ ]:
dietrich.setBasis(allowedGenes=dietrichGenes, maxSamples=7500, includeCriteria=dietrich.annotations.isin(["AT2", "AT1", "Aberrant basaloid", "Basal", "MCC", "Goblet", "RASC"]))
dietrich.basis.to_csv("/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/OutsidePaperObjects/Bases/DietrichRAS7500ANOVA2.csv")

In [ ]:
includeCriteria = dietrich.annotations.isin(["AT2", "AT1", "Aberrant basaloid", "Basal", "MCC", "Goblet", "RASC"])
dietrich.testBasis(allowedGenes=dietrichGenes, maxBasisSamples=5000, maxTestSamples=3000, includeCriteria=includeCriteria, trialCount=3, seed=3)

In [ ]:
SimilarityHelper.plotBasisTestConfusionMatrix(dietrich, title="Dietrich Reclassifications", decimalMode="Clean",
                                              # outFile="../../Results/BasisTesting/Dietrich Basis Confusion Matrix Downsampled Basis 2500 Test 1000 Trials 1 Clean No Diff AT2.png"
)

In [ ]:
dietrich.project(Natri, "Natri")
# dietrich.project(Natri2000, "Natri2000")

In [ ]:
# includeCriteria = bharat.metadata["Sample Status"] == "Control"
dietrichSimilarityMap = SimilarityHelper.getMatchingProjections(dietrich, "Natri", includeCriteria=None)
SimilarityHelper.similarityBoxplot(dietrichSimilarityMap, title="Dietrich vs Natri", 
                                   outFile="../../PendingResults/Dietrich vs Natri Boxplot.png"
)

In [ ]:
includeCriteria = dietrich.annotations.isin(["AT1", "AT2", "Basal", "Aberrant basaloid", "Aberrant basal", "Transitional AT2"])
# axis1 = 'KRT5-KRT17+'
axis1 = 'AT1'
axis2 = 'AT2'

ax = SimilarityHelper.plotTwo(dietrich, "Natri", axis1, axis2,
                         title="Dietrich projected on Natri", DPI=300, legendInner=True, axisRenames=("Natri AT1 Cell Score", "Natri AT2 Cell Score"),
                         outFile="../../PendingResults/Dietrich vs Natri AT1 vs AT2 Downsampled 8000.png",
                         includeCriteria=includeCriteria, maxLabelCount=8000,
                         unsupervisedContour=False
)

# Sankar

In [ ]:
sankarMUC5B = TopObject.TopObject("SankarMUC5B", skipProcess=True)
sankarMUC5B.metadata

In [ ]:
sankarMUC5B.annotations.value_counts()

In [ ]:
sankarMUC5B.setBasis(includeCriteria=sankarMUC5B.metadata[sankarMUC5B.timeColumn] != "THD077-NT")

In [ ]:
caseBasis = sankarMUC5B.basis
# controlBasis = sankarMUC5B.basis

In [ ]:
## (caseBasis["MUC5B-stressecreting"] - controlBasis["AT2-like"]).sort_values(ascending=False).iloc[:50]

In [ ]:
sankarMUC5B.project(Natri, "Natri")

In [ ]:
includeCriteria = None
SankarSimilarityMapLungMAP = SimilarityHelper.getMatchingProjections(sankarMUC5B, "Natri", includeCriteria=includeCriteria)
SimilarityHelper.similarityBoxplot(SankarSimilarityMapLungMAP, title="Sankar vs Natri", 
                                   outFile="../../PendingResults/SankarMUC5B vs Natri Boxplot.png"
)

In [ ]:
# includeCriteria = ~sankarMUC5B.annotations.isin(["PNEC", "Ciliated", "Goblet", "Club", "Secretory"])
ax = SimilarityHelper.plotTwo(sankarMUC5B, "Natri", "KRT5-KRT17+", "AT2",
                         maxLabelCount=500, #unsupervisedContour=True, 
                         includeCriteria=None, #gene="IL32",
                         title="Sankar Projected Onto Natri Reference", 
                         # outFile="../../PendingResults/SankarMUC5B vs Natri SKAR vs AT2 Downsampled 500.png",
)

# Overlays

In [ ]:
## Load TopObjects
bharat = TopObject.TopObject("Bharat", skipProcess=False)
# habermann = TopObject.TopObject("Habermann", skipProcess=False)
kaminski2020 = TopObject.TopObject("Adams2020", skipProcess=False, keep=True)
kaminski2020.name = "Adams"
PPFE = TopObject.TopObject("PPFE", skipProcess=False)
PPFE.name = "Ruwisch"
# natri = TopObject.TopObject("Natri", skipProcess=False, maxSamples=10000, keepFull=["AT2"])

In [ ]:
lauren = TopObject.TopObject("Lauren_06_24_25", skipProcess=True)
lauren.name = "Ayers"

In [ ]:
# Reannotate with new labels   TODO: use some function to map dicts of labels for less cumbersome approach
# newAnnotations = [val if not val in ["KRT17+ KRT5-"] else "SKAR" for val in bharat.annotations]
# bharat.anndata.obs["celltypeNew"] = newAnnotations
# newAnnotations = [val if val != "Aberrant_Basaloid" else "SKAR" for val in kaminski2020.annotations]
# newAnnotations = [val if val != "ATI" else "AT1" for val in newAnnotations]
# newAnnotations = [val if val != "ATII" else "AT1" for val in newAnnotations]
# kaminski2020.anndata.obs["celltypeNew"] = newAnnotations
# newAnnotations = [val if not val in ["Aberrant_Basaloid"] else "SKAR" for val in PPFE.annotations]
# newAnnotations = [val if not val in ["AEC_intermediate"] else "Differentiating AT2" for val in newAnnotations]
# newAnnotations = [val if val != "AEC1" else "AT1" for val in newAnnotations]
# newAnnotations = [val if val != "AEC2" else "AT2" for val in newAnnotations]
# PPFE.anndata.obs["celltypeNew"] = newAnnotations
# # newAnnotations = [val if val != "KRT5-KRT17+" else "SKAR" for val in natri.annotations]
# # natri.anndata.obs["celltypeNew"] = newAnnotations
# newAnnotations = [val if val != "KRT5-/KRT17+" else "SKAR" for val in habermann.annotations]
# newAnnotations = [val if not val in ["Transitional AT2"] else "Differentiating AT2" for val in newAnnotations]
# habermann.anndata.obs["celltypeNew"] = newAnnotations


# topObjects = [kaminski2020, bharat, PPFE, natri, habermann]
topObjects = [kaminski2020, bharat, PPFE, habermann]
for topObject in topObjects:
    topObject.cellTypeColumn = "celltypeNew"
    topObject.setMetadata()

In [ ]:
## Load bases
# HaberMAP = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="HaberMAP500ANOVA2", #HaberMAPANOVA1
#                                          basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet", "Secretory", "KRT5-/KRT17+"]
# )
# Adams = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="Adams500", #Adams500ANOVA2
#                                          #basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet", "Secretory", "KRT5-/KRT17+"]
# )
HaberMAP = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="HaberMAP500", #HaberMAPANOVA1
                                         basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet", "Secretory", "KRT5-/KRT17+"]
)
# lungMAP = SimilarityHelper.loadBasis(basisCollection=basisCollection, basisName="LungMAPANOVA",
#                                          basisKeep=["AT2", "AT1", "Basal", "Ciliated", "Goblet"]#, "Secretory"]
# )
# basisName = "HaberMAP"
# basisName = "LungMAP + Habermann"
# basis = HaberMAP
# basis = habermann.combineBases(lungMAP, firstKeep=["KRT5-/KRT17+"], name=basisName)
# basis = habermann.combinedBases[basisName]

In [ ]:
## Overlay within same dataset
topObjects = [kaminski2020]
diseaseCategory = "Disease_Identity"
# diseaseCategory = "Diagnosis"
# names = ["IPF", "cHP", "sacroidosis", "NSIP"]
# additionalCriteriaOthers=~topObject.annotations.isin(["Basal", "AT1", "AT2"])
names = ["IPF", "COPD"]
additionalCriteriaOthers = ~topObjects[0].annotations.isin(["Club", "Goblet", "Ciliated"]) #"ATII", "ATI"
additionalCriteriaAll = ~topObjects[0].annotations.isin(["Ciliated", "Club", "Goblet"])
# additionalCriteriaOthers = None
includeCriteriaList = SimilarityHelper.setupIncludeCriteria(topObjects[0], diseaseCategory, names, additionalCriteriaOthers=additionalCriteriaOthers, additionalCriteriaAll=additionalCriteriaAll)
otherProjections, otherAnnotations = SimilarityHelper.setupOverlay(topObjects, "LungMAP", includeCriteriaList, basis=lungMAP)

In [ ]:
## Overlay different datasets
includeCriteriaList = []
# includeCriteriaList.append(kaminski2020.annotations.isin(["SKAR"])) # Aberrant_Basaloid
# includeCriteriaList.append(bharat.annotations.isin(["SKAR"])) # KRT17+ KRT5-
includeCriteriaList.append(PPFE.annotations.isin(["Differentiating AT2"])) # Aberrant_Basaloid
# includeCriteriaList.append(natri.annotations.isin(["SKAR"]))
includeCriteriaList.append(habermann.annotations.isin(["Differentiating AT2", "AT2", "AT1"])) # KRT5-/KRT17+

topObjects = [PPFE, habermann]
# topObjects = [kaminski2020, bharat, PPFE, habermann]
# topObjects = [kaminski2020, bharat, PPFE, natri]
# otherProjections, otherAnnotations, names = SimilarityHelper.setupOverlay(topObjects, "LungMAP", includeCriteriaList, basis=lungMAP, forceProject=False)
otherProjections, otherAnnotations, names = SimilarityHelper.setupOverlay(topObjects, "Natri", includeCriteriaList, basis=Natri, forceProject=False)

In [ ]:
ax = SimilarityHelper.plotTwo(kaminski2020, "LungMAP", "Basal", "AT1",
             additionalProjections=otherProjections, additionalAnnotations=otherAnnotations, name=names[0], additionalNames=names[1:],
             includeCriteria=includeCriteriaList[0],
             title="Adams IPF, Habermann IPF, Bharat COVID-19, and Ruwisch PPFE projected on HaberMAP", axisFontSize=18, legendFontSize=13, legendInner=True, titleFontSize=14,
             # title="Kaminski + Bharat projected onto LungMAP + Habermann Reference",
             # outFile="../Results/Overlays/Kaminski + Habermann AT1 vs AT2 Downsampled 500.png",
             # outFile="../Results/Overlays/Overlay Kaminski, Habermann, Bharat, and PPFE vs LungMAP Basal vs AT2 Downsampled 500 (With SKAR).png",
             unsupervisedContour=False, maxLabelCount=500,
)

In [ ]:
# rng = np.random.default_rng(1)
# list(rng.choice(len(bharat.annotations), size=10, replace=False))
# curr = bharat.annotations[bharat.annotations == "AT2"]
# curr.iloc[[0, 1]]
list(curr.index[rng.choice(len(curr), size=3, replace=False)])

In [ ]:
# ax = SimilarityHelper.plotTwo(kaminski2020, "HaberMAP", "KRT5-/KRT17+", "AT2",
#              additionalProjections=otherProjections, additionalAnnotations=otherAnnotations, name=names[0], additionalNames=names[1:],
#              includeCriteria=includeCriteriaList[0],
#              title="Adams IPF, Bharat COVID-19, and PPFE projected on HaberMAP", axisFontSize=18, legendFontSize=13, legendInner=True, titleFontSize=15,
#              # outFile="../Results/Overlays/Overlay Adams, Bharat, and PPFE vs HaberMAP 500 Basal vs SKAR Downsampled 500.png",
#              unsupervisedContour=False, maxLabelCount=500,
# )

overlayLabels = ["Habermann AT2", "Habermann AT1", "Ruwisch Differentiating AT2", "Habermann Differentiating AT2"]
# overlayLabels = ["Habermann Basal", "Bharat SKAR", "Adams SKAR", "Ruwisch SKAR", "Habermann SKAR"]
# overlayLabels = ["Habermann AT2", "Habermann KRT5-/KRT17+", "Bharat KRT17+ KRT5-", "Ruwisch Aberrant_Basaloid", "Adams Aberrant_Basaloid"]
# for i in range(len(overlayLabels)):
for i in range(len(overlayLabels) - 1, len(overlayLabels)):
    ax = SimilarityHelper.plotTwo(PPFE, "Natri", "AT1", "AT2",
                 additionalProjections=otherProjections, additionalAnnotations=otherAnnotations, name=names[0], additionalNames=names[1:],
                 includeCriteria=includeCriteriaList[0], overlayLabels=overlayLabels[:i+1], labelDimensions=True, axisRenames=("Natri AT1 Cell Score", "Natri AT2 Cell Score"),
                 # axisFontSize=18, legendFontSize=16, titleFontSize=12, 
                 DPI=300, title="",#"Adams IPF, Bharat COVID-19, Ruwisch PPFE, and Habermann ILDs projected on Natri",
                 # outFile="../../Results/Overlays/Overlay Adams, Bharat, PPFE, and Habermann vs Natri 2000 ANOVA3 AT2 vs SKAR Downsampled 500 (" + str(i) + ").png",
                 outFile="../../Results/Overlays/Overlay PPFE and Habermann vs Natri 2000 ANOVA3 AT1 vs AT2 Downsampled 2000 Diff AT2.png",
                 unsupervisedContour=False, maxLabelCount=2000,
    )

In [ ]:
# npIndices = np.concatenate(indices)
# len(npIndices)
annotations[npIndices]

In [ ]:
## Overlay different datasets
includeCriteriaList = []
includeCriteriaList.append(lauren.annotations.isin(["ABI2", "ABI1", "iAT2"]))
includeCriteriaList.append(sankarMUC5B.annotations.isin(["AT2-like", "MUC5B-stressecreting"]))

topObjects = [lauren, sankarMUC5B]
otherProjections, otherAnnotations, names = SimilarityHelper.setupOverlay(topObjects, "Natri", includeCriteriaList, basis=Natri, forceProject=False)

In [ ]:
ax = SimilarityHelper.plotTwo(kaminski2020, "HaberMAP", "KRT5-/KRT17+", "AT2",
             additionalProjections=otherProjections, additionalAnnotations=otherAnnotations, name=names[0], additionalNames=names[1:],
             includeCriteria=includeCriteriaList[0],
             title="Adams IPF, Bharat COVID-19, and PPFE projected on HaberMAP", axisFontSize=18, legendFontSize=13, legendInner=True, titleFontSize=15,
             # outFile="../Results/Overlays/Overlay Adams, Bharat, and PPFE vs HaberMAP 500 Basal vs SKAR Downsampled 500.png",
             unsupervisedContour=False, maxLabelCount=500,
)

# Combined Pathway Analysis

In [ ]:
includeCriteria = habermann.annotations.isin(["AT1", "AT2", "Basal", "SKAR", "Differentiating AT2"])
habermann.setBasis(includeCriteria=includeCriteria)
natri.setBasis()
habermann.basis.columns = [habermann.name + " " + col for col in habermann.basis.columns]
natri.basis.columns = [natri.name + " " + col for col in natri.basis.columns]
natri.basis

In [ ]:
target = "Habermann SKAR"
comparator = "Natri Basal"
diseaseColumn = "Diagnosis"
haberNatri = habermann.combineBases(natri.basis)
# includeCriteria = np.logical_or(habermann.annotations == target, np.logical_and(habermann.annotations == comparator, habermann.metadata[diseaseColumn] != "Control"))
diffTableMap = CriticalityHelper.getDifferentiallyExpressedGenes(None, habermann.cellTypeColumn, target, 
                        individualCompare=True, includeCriteria=None, basis=haberNatri
)

habermannTargetDF = CriticalityHelper.getCombinedTopGenes(diffTableMap, habermann.processed,
                        useBasis=True, includeCriteria=habermann.annotations == target, missesAllowed=3, minimumChange=0.0001, minimumQuantileChange=0.7, maximumPVal=0.5,
                        expressionThreshold=0.0001, diffType="scores", checkSurface=False, requireOverexpression=False, #secondDF=natri.df.loc[:, natri.annotations == "SKAR"],
                        # outFile="../../PendingResults/Habermann Differential Cell Surface Genes.csv"
)
habermannTargetDF

In [ ]:
pre_res = gs.prerank(rnk=habermannTargetDF.loc[habermannTargetDF["Successes"] == 1][comparator],
                     gene_sets='Reactome_Pathways_2024', #KEGG_2026 GO_Biological_Process_2025
                     threads=4,
                     min_size=5,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True, # see what's going on behind the scenes
                    )
# pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/Habermann SKAR vs Disease " + comparator + " Reactome Pathways.csv")
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :].to_csv("../../Results/Pathways/Habermann Disease vs Control " + cellState + " Reactome Pathways.csv")
pre_res.res2d.loc[pre_res.res2d["FDR q-val"] < 0.1, :]

In [ ]:
library = "Reactome"
location = "/restricted/projectnb/crem-trainees/Kotton_Lab/Eitan/Results/Pathways/"
names = ["Habermann", "Adams", "Natri", "Bharat", "Ruwisch"]
pathways = {"Disease vs Control": {}, "SKAR vs Control": {}, "SKAR vs Disease": {}}
cellStates = ["AT1", "AT2", "Differentiating AT2", "SKAR", "Basal"]
for comparison in pathways.keys():
    for state in cellStates:
        pathways[comparison][state] = {}
        for name in names:
            fileName = location + name + " " + comparison + " " + state + " " + library + " Pathways.csv"
            if os.path.exists(fileName):
                pathways[comparison][state][name] = pd.read_csv(fileName, index_col="Term")
pathways["SKAR vs Control"]["AT2"]["Habermann"]

In [ ]:
# Skar vs Disease = 16 comparisons; SKAR vs Control = 15, Disease vs Control = 14
pd.set_option('display.float_format', '{:.3f}'.format)
comparison = "Disease vs Control"
# comparison = "Disease vs Control"
# states = ["Differentiating AT2"]
states = ["AT2", "AT1", "Basal", "Differentiating AT2"]
simplified = False
# file = "../../Results/Pathways/NewPathways/" + comparison + " " + states[0] + " Reactome Pathways Summary" + (" Simplified.csv" if simplified else " Stringent.csv")
file = "../../Results/Pathways/NewPathways/" + comparison + " Full Reactome Pathways Summary" + (" Simplified.csv" if simplified else " Stringent.csv")
pathwaySummary = CriticalityHelper.getPathwaySummary(pathways, comparisons=[comparison], states=states, names=None, simplified=simplified, significance=0.05, tolerance=0.7, outFile=file)
print((lebn(pathwaySummary.columns) - 4) / 3)
pathwaySummary.iloc[:30]

In [ ]:
# sorted_dict = dict(sorted(pathwayHits.items(), key=lambda item: item[1], reverse=True))
# pathwayHitsFrame = pd.DataFrame.from_dict(pathwayHits, orient="index")
# pathwayHitsFrame = pathwayHitsFrame.rename(columns={0: "Hits"})
# pathwayHitsFrame.sort_values("Hits", ascending=False).head(16)
pathwaysFrame.sort_values("Hits", ascending=False).to_csv("../../PendingResults/SKAR vs Control Pathways Top Genes.csv")
# pathwayHitsFrame.sort_values("Hits", ascending=False).to_csv("../../PendingResults/Human Pathways Summary.csv")

In [ ]:
merged

In [ ]:
# from adjustText import adjust_text
# x = pd.read_csv("../../Results/Pathways/NewPathways/SKAR vs Disease Full Reactome Pathways Summary.csv", index_col="Unnamed: 0")
# y = pd.read_csv("../../Results/Pathways/NewPathways/Disease vs Control Full Reactome Pathways Summary.csv", index_col="Unnamed: 0")
# x.index.name = y.index.name = "Pathway"
# metric = "Hits"
# merged = pd.merge(x[metric] / 16, y[metric] / 14, how="outer", on="Pathway", suffixes=(" Aberrant vs Disease", " Disease vs Control")).fillna(0)
# labels = [pathway for pathway in merged.index if merged.loc[pathway, metric + " Aberrant vs Disease"] > 0.6]
# newX, newY = (merged[metric + " Aberrant vs Disease"], merged[metric + " Disease vs Control"])
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
# for pathway, (currentX, currentY) in zip(labels, zip(newX[merged.index.isin(labels)], newY[merged.index.isin(labels)])):
#     ax.text(currentX, currentY, pathway, fontsize=12)
plot = sns.scatterplot(x=newX, y=newY, ax=ax)
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.1, 0.85)
texts = []
for pathway in labels:
    texts.append(
        ax.text(
            merged.loc[pathway, metric + " Aberrant vs Disease"],
            merged.loc[pathway, metric + " Disease vs Control"],
            pathway,
            fontsize=12
        )
    )
adjust_text(
    texts,
    ax=ax,
    expand_points=(1.2, 1.5),
    expand_text=(1.3, 1.6),
    only_move={'points': '', 'text': 'xy'},
    force_points=0.5,
    force_text=1.2,
    # force_pull=2.5,
    pull_threshold=50,
    max_move=200,
    iter_lim=2000,
    prevent_crossings=True,
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.5, shrinkA=10)
)
ax.set_xlabel("Pathway Occurrence in Aberrant vs Disease", fontsize=14)
ax.set_ylabel("Pathway Occurrence in Disease vs Control", fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=10)

plt.savefig("../../PendingResults/Pathway Analysis Comparison (SKAR vs Disease) vs (Disease vs Control) FDR 0.1.png")
plt.show()